# Reproduce the Part 1 experiments

This notebook embeds the complete implementation and helper scripts. It can recreate them in an isolated `reproduction` directory; original results remain unchanged. Full search is opt-in because it runs hundreds of fits. The report notebook executes directly from saved evidence. No deep learning or external trial records are used.

Install the locked Python 3.12 environment using the commands in `READ_ME_FIRST.txt`. On macOS the XGBoost/LightGBM screen needs libomp. The selected model does not. The first validation result remains separate from the later unvalidated full-fit candidate.

In [1]:
from pathlib import Path
import shutil, subprocess, sys, json
root = Path.cwd()
replica = root / 'reproduction'
replica.mkdir(exist_ok=True)
(replica/'experiments').mkdir(exist_ok=True)
(replica/'predictions').mkdir(exist_ok=True)
shutil.copytree(root/'data', replica/'data', dirs_exist_ok=True)


PosixPath('/Users/matthew/Desktop/AMLMidterm/midterm/reproduction/data')

### audit_candidates.py

Complete source:

In [2]:
source = '"""Nested shortlist audit and temporal stress test. Does NOT open validation or test labels."""\nimport experiment as e\nimport numpy as np,json,time\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nrank=json.loads(Path(\'experiments/robust-ranking.json\').read_text())\nens=json.loads(Path(\'experiments/ensemble-ranking.json\').read_text())\nssl=next(x[\'config\'] for x in rank if x[\'config\'][\'model\']==\'pseudo\')\nssl_ensemble=next(x[\'config\'] for x in ens if len(x[\'config\'][\'members\'])==3 and x[\'config\'][\'members\'][0][\'model\']==\'svc\')\ncandidates=[dict(model=\'logistic\',C=.1,scale=\'global\'),dict(model=\'logistic\',C=.3,scale=\'global\'),dict(model=\'svc\',C=10,gamma=.001,scale=\'standard\'),dict(model=\'svc\',C=10,gamma=.001,scale=\'standard\',features=[\'drug_targets\'],feature_weight=1.),ssl,ens[0][\'config\'],ssl_ensemble]\ne.save_json(\'experiments/nested-candidates.json\',candidates)\nouter=e.make_splits(seed=6172026,n_splits=5);pred=np.zeros_like(e.Y);prob=np.zeros_like(e.Y,float);records=[]\nwith threadpool_limits(limits=1):\n for fold,(a,b) in enumerate(outer):\n  print(\'OUTER\',fold,flush=True);inner=e.make_splits(e.Y[a],e.GROUP[e.LAB[a]],seed=1200+fold,n_splits=3)\n  pool=[]\n  for ci,c in enumerate(candidates):\n   ip=np.zeros((len(a),5))\n   for ia,ib in inner:\n    exclude=np.r_[e.GROUP[e.LAB[b]],e.GROUP[e.LAB[a[ib]]]]\n    ip[ib]=e.predict(a[ia],e.DATA.iloc[e.LAB[a[ib]]],c,excluded_groups=exclude)\n   rule,score=e.choose_rule(e.Y[a],ip);pool.append(dict(config=c,rule=rule,score=score))\n   print(\'INNER\',fold,ci,score,rule,flush=True)\n  # A small tie favors an earlier, simpler candidate.\n  maximum=max(r[\'score\'] for r in pool);best=next(r for r in pool if r[\'score\']>=maximum-.003)\n  pp=e.predict(a,e.DATA.iloc[e.LAB[b]],best[\'config\'],excluded_groups=e.GROUP[e.LAB[b]])\n  prob[b]=pp;pred[b]=e.decisions(pp,best[\'rule\'])\n  rec=dict(fold=fold,selected=best,inner=pool,outer=e.metrics(e.Y[b],pp,best[\'rule\']))\n  records.append(rec);e.save_json(\'experiments/nested-progress.json\',records);print(\'OUTER RESULT\',rec[\'outer\'],flush=True)\n from sklearn.metrics import f1_score\n result=dict(sample_f1=f1_score(e.Y,pred,average=\'samples\'),macro_f1=f1_score(e.Y,pred,average=\'macro\'),per_label=dict(zip(e.LABELS,f1_score(e.Y,pred,average=None))),folds=records,limitation=\'Conditional shortlist audit. Candidate shortlist was informed by full-training exploratory CV; not an unbiased nested rerun of the complete adaptive search. External validation remains unopened.\')\n e.save_json(\'experiments/nested-audit.json\',result);np.savez_compressed(\'experiments/nested-predictions.npz\',p=prob,pred=pred,y=e.Y)\n # Chronology tests are deliberately hard: most extra rare-reason labels occur in later years.\n temporal=[]\n for cutoff in [2002,2005]:\n  years=e.DATA.iloc[e.LAB].start_year.to_numpy();a=np.flatnonzero(years<=cutoff);b=np.flatnonzero(years>cutoff)\n  a=a[~np.isin(e.GROUP[e.LAB[a]],e.GROUP[e.LAB[b]])]\n  for c in candidates:\n   pp=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])\n   # Fixed default decision: no tuning on temporal holdout.\n   rec=dict(cutoff=cutoff,config=c,n_train=len(a),n_test=len(b),train_positive=e.Y[a].sum(0).tolist(),test_positive=e.Y[b].sum(0).tolist(),**e.metrics(e.Y[b],pp,\'t0.5\'))\n   temporal.append(rec);print(\'TEMPORAL\',rec,flush=True)\n e.save_json(\'experiments/temporal-audit.json\',temporal)\nprint(\'DONE\',flush=True)\n'
(replica / 'audit_candidates.py').write_text(source)


3379

### experiment.py

Complete source:

In [3]:
source = '"""Reproducible, train-only Part 1 development engine. See experiments/protocol.txt."""\nimport os\nfor var in [\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'VECLIB_MAXIMUM_THREADS\',\'NUMEXPR_NUM_THREADS\']:\n    os.environ.setdefault(var,\'1\')\nimport argparse, hashlib, json, time, traceback, warnings, re, ast\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.special import expit, softmax\nfrom scipy.special import logit\nfrom scipy.sparse.linalg import spsolve\nfrom sklearn.preprocessing import StandardScaler, normalize\nfrom sklearn.decomposition import PCA, TruncatedSVD\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression, Ridge\nfrom sklearn.kernel_ridge import KernelRidge\nfrom sklearn.gaussian_process import GaussianProcessClassifier\nfrom sklearn.gaussian_process.kernels import RBF, ConstantKernel\nfrom sklearn.neighbors import NeighborhoodComponentsAnalysis\nfrom sklearn.svm import SVC, LinearSVC\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.neighbors import KNeighborsRegressor, NearestNeighbors\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis\nfrom sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier\nfrom sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, DBSCAN, HDBSCAN, SpectralClustering, Birch\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.metrics import f1_score, pairwise_distances\nfrom sklearn.multioutput import ClassifierChain\nfrom sklearn.cross_decomposition import PLSRegression\nfrom scipy.special import logsumexp\nfrom scipy.linalg import cho_factor, cho_solve\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom threadpoolctl import threadpool_limits\nwarnings.filterwarnings(\'ignore\',category=FutureWarning)\nROOT=Path(__file__).resolve().parent\nLABELS=[\'Safety\',\'Funding\',\'Efficacy\',\'Recruitment\',\'Other\']; EMB=[f\'emb_{i}\' for i in range(150)]\nDATA=pd.read_csv(ROOT/\'data/part1_train.csv\')\nLAB=np.flatnonzero(DATA[LABELS].ge(0).all(axis=1)); UNLAB=np.flatnonzero(DATA[LABELS].eq(-100).all(axis=1))\nY=DATA.iloc[LAB][LABELS].to_numpy(int)\nGROUP=pd.util.hash_pandas_object(DATA[EMB].round(10),index=False).to_numpy()\nassert len(LAB)+len(UNLAB)==len(DATA)\nCACHE={}\n\ndef config_id(c): return hashlib.sha256(json.dumps(c,sort_keys=True).encode()).hexdigest()[:14]\ndef save_json(path,obj): Path(path).write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))\ndef make_splits(y=Y,groups=None,seed=20261104,n_splits=5):\n    if groups is None: groups=GROUP[LAB]\n    unique,inv=np.unique(groups,return_inverse=True)\n    gy=np.array([y[inv==i].max(axis=0) for i in range(len(unique))])\n    return [(np.flatnonzero(np.isin(inv,a)),np.flatnonzero(np.isin(inv,b))) for a,b in MultilabelStratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed).split(gy,gy)]\n\ndef decisions(p,rule=\'t0.5\'):\n    p=np.clip(np.asarray(p),0,1)\n    if rule==\'expected_f1\':\n        patterns=((np.arange(32)[:,None]>>np.arange(5))&1)\n        prob=np.prod(np.where(patterns[None,:,:],p[:,None,:],1-p[:,None,:]),axis=2)\n        choices=patterns[1:]\n        f1=2*(choices@patterns.T)/(choices.sum(1)[:,None]+patterns.sum(1)[None,:])\n        return choices[np.argmax(prob@f1.T,axis=1)]\n    if rule==\'argmax\':\n        out=np.zeros_like(p,dtype=int);out[np.arange(len(p)),p.argmax(1)]=1;return out\n    out=(p>=float(rule[1:])).astype(int)\n    empty=out.sum(1)==0\n    out[np.flatnonzero(empty),p[empty].argmax(1)]=1\n    return out\nRULES=[\'t0.25\',\'t0.35\',\'t0.45\',\'t0.5\',\'t0.55\',\'expected_f1\',\'argmax\']\ndef metrics(y,p,rule=\'t0.5\'):\n    pred=decisions(p,rule)\n    return {\'sample_f1\':float(f1_score(y,pred,average=\'samples\',zero_division=0)), \'macro_f1\':float(f1_score(y,pred,average=\'macro\',zero_division=0)),**{f\'f1_{l}\':float(v) for l,v in zip(LABELS,f1_score(y,pred,average=None,zero_division=0))}}\ndef choose_rule(y,p):\n    scores=[metrics(y,p,r)[\'sample_f1\'] for r in RULES]\n    # Prefer default threshold when differences are negligible.\n    best=max(scores); idx=min([i for i,s in enumerate(scores) if s>=best-1e-12],key=lambda i:abs(i-3))\n    return RULES[idx],best\n\ndef preprocess(A,B,c):\n    A=np.asarray(A,float);B=np.asarray(B,float)\n    mode=c.get(\'scale\',\'standard\')\n    if mode==\'standard\':\n        sc=StandardScaler();A=sc.fit_transform(A);B=sc.transform(B)\n    elif mode==\'global\':\n        mean=A.mean(0);std=np.sqrt(np.mean(np.var(A,axis=0)));A=(A-mean)/std;B=(B-mean)/std\n    elif mode==\'raw\':\n        # Global scale without centering preserves cosine and original relative geometry.\n        std=np.sqrt(np.mean(np.var(A,axis=0)));A=A/std;B=B/std\n    n=c.get(\'pca\',0)\n    if n:\n        pc=PCA(n_components=min(n,A.shape[1],A.shape[0]-1),svd_solver=\'full\');A=pc.fit_transform(A);B=pc.transform(B)\n        power=c.get(\'white\',0)\n        if power:\n            scale=np.maximum(pc.explained_variance_,1e-8)**(power/2);A/=scale;B/=scale\n    if c.get(\'norm\',False):A=normalize(A);B=normalize(B)\n    return A,B\n\ndef chemical(frame,kind):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import Descriptors, rdFingerprintGenerator\n    RDLogger.DisableLog(\'rdApp.*\')\n    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)\n    desc=[Descriptors.MolWt,Descriptors.MolLogP,Descriptors.TPSA,Descriptors.NumHDonors,Descriptors.NumHAcceptors,Descriptors.NumRotatableBonds,Descriptors.RingCount,Descriptors.FractionCSP3,Descriptors.HeavyAtomCount]\n    out=[]\n    for field in frame.drug_smiles.fillna(\'\'):\n        mols=[m for s in field.split(\';\') if s.strip() and (m:=Chem.MolFromSmiles(s.strip())) is not None]\n        if kind==\'fingerprint\':v=np.max([gen.GetFingerprintAsNumPy(m) for m in mols],axis=0) if mols else np.zeros(1024)\n        else:\n            z=np.array([[fn(m) for fn in desc] for m in mols])\n            v=np.r_[len(mols),z.mean(0),z.max(0)] if mols else np.zeros(19)\n        out.append(v)\n    return np.array(out,float)\n\ndef block_features(A,B,block):\n    if block in (\'fingerprint\',\'descriptors\'):\n        a=chemical(A,block);b=chemical(B,block)\n    elif block==\'year\':a=A[[\'start_year\']].to_numpy();b=B[[\'start_year\']].to_numpy()\n    elif block==\'missingness\':\n        cols=[\'criteria\',\'diseases_cleaned\',\'drugs\',\'icdcodes\',\'matched_drugs\',\'drug_targets\',\'drug_actions\',\'drug_pathways\',\'drug_smiles\']\n        a=A[cols].isna().astype(float).to_numpy();b=B[cols].isna().astype(float).to_numpy()\n    elif block==\'counts\':\n        cols=[\'criteria\',\'diseases_cleaned\',\'drugs\',\'icdcodes\',\'matched_drugs\',\'drug_targets\',\'drug_actions\',\'drug_pathways\',\'drug_smiles\']\n        def counts(D):return np.column_stack([np.log1p(D[c].fillna(\'\').str.len()) for c in cols]+[D[c].fillna(\'\').str.count(\';\')+D[c].notna() for c in cols[4:]])\n        a=counts(A);b=counts(B)\n    elif block.startswith(\'pathway_level\'):\n        catalog=pd.read_csv(ROOT/\'data/pathway_catalog.csv\');levels=dict(zip(catalog.reactome_id,catalog.pathway_level))\n        if block==\'pathway_levels\':\n            def calc(D):return np.array([[sum(levels.get(s.strip(),-1)==i for s in str(v).split(\';\')) for i in range(10)] for v in D.drug_pathways.fillna(\'\')],float)\n            a=np.log1p(calc(A));b=np.log1p(calc(B))\n        else:\n            lev=int(block[-1]); allowed=set(catalog.loc[catalog.pathway_level<=lev,\'reactome_id\'])\n            def clean(D):return D.drug_pathways.fillna(\'\').map(lambda x:\' \'.join(t.strip() for t in x.split(\';\') if t.strip() in allowed))\n            v=TfidfVectorizer(token_pattern=r\'[^\\s]+\',min_df=2,max_features=1000);a=v.fit_transform(clean(A)).toarray();b=v.transform(clean(B)).toarray();return a,b\n    else:\n        col=block.replace(\'_char\',\'\');char=block.endswith(\'_char\')\n        v=TfidfVectorizer(lowercase=True,analyzer=\'char_wb\' if char else \'word\',ngram_range=(3,5) if char else (1,2),min_df=2,max_df=0.99,max_features=3000,sublinear_tf=True,token_pattern=r\'(?u)\\b[\\w-]+\\b\')\n        a=v.fit_transform(A[col].fillna(\'\'));b=v.transform(B[col].fillna(\'\'))\n        # Keep exact sparse-text geometry; at this dataset size dense blocks fit in memory.\n        return a.toarray(),b.toarray()\n    if block!=\'fingerprint\':\n        sc=StandardScaler();a=sc.fit_transform(a);b=sc.transform(b)\n    a=np.nan_to_num(a);b=np.nan_to_num(b)\n    norm=np.sqrt(np.mean(np.sum(a*a,axis=1)))\n    return a/max(norm,1e-8),b/max(norm,1e-8)\n\ndef features(train_idx,query_frame,c):\n    fitframe=DATA.iloc[train_idx]\n    A,B=preprocess(fitframe[EMB],query_frame[EMB],c)\n    blocks=c.get(\'features\',[])\n    if blocks:\n        base_norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))\n        weight=c.get(\'feature_weight\',0.3)\n        for block in blocks:\n            a,b=block_features(fitframe,query_frame,block)\n            A=np.column_stack([A,a*base_norm*weight]);B=np.column_stack([B,b*base_norm*weight])\n    return A,B\n\nclass Graph:\n    """Normalized graph diffusion of all five binary labels; independent negative mass implicit."""\n    def __init__(self,c):self.c=c\n    def fit(self,X,Y):\n        self.X=X; n=len(X);c=self.c;k=min(c.get(\'k\',15),n-1);self.k=k\n        self.nn=NearestNeighbors(n_neighbors=k+1).fit(X);dist,ix=self.nn.kneighbors(X)\n        dist=dist[:,1:];ix=ix[:,1:];self.sigma=np.maximum(dist[:,-1],1e-8)\n        self.band=float(np.median(dist[:,-1]))*c.get(\'band\',1.)\n        if c.get(\'graph\',\'adaptive\')==\'adaptive\':w=np.exp(-dist**2/(self.sigma[:,None]*self.sigma[ix]+1e-12))\n        elif c.get(\'graph\')==\'binary\':w=np.ones_like(dist)\n        else:w=np.exp(-dist**2/max(self.band**2,1e-12))\n        W=sparse.csr_matrix((w.ravel(),(np.repeat(np.arange(n),k),ix.ravel())),shape=(n,n))\n        W=(W+W.T)*0.5\n        degree=np.maximum(np.asarray(W.sum(1)).ravel(),1e-12)\n        known=Y[:,0]>=0\n        if c.get(\'hard\',False):\n            P=sparse.diags(1/degree)@W;u=np.flatnonzero(~known);l=np.flatnonzero(known)\n            alpha=c.get(\'alpha\',0.99)\n            U=spsolve(sparse.eye(len(u),format=\'csc\')-alpha*P[u][:,u].tocsc(),alpha*P[u][:,l]@Y[l]+(1-alpha)*Y[l].mean(0))\n            self.F=Y.copy().astype(float);self.F[u]=U\n        else:\n            S=sparse.diags(degree**-0.5)@W@sparse.diags(degree**-0.5)\n            Z=np.zeros((n,6));Z[known,:5]=Y[known];Z[known,5]=1\n            sol=spsolve(sparse.eye(n,format=\'csc\')-c.get(\'alpha\',0.95)*S.tocsc(),Z)\n            self.F=np.divide(sol[:,:5],sol[:,5,None],out=np.tile(Y[known].mean(0),(n,1)),where=sol[:,5,None]>1e-15)\n        self.F=np.clip(self.F,0,1);return self\n    def predict(self,Q):\n        d,ix=self.nn.kneighbors(Q,n_neighbors=self.k)\n        if self.c.get(\'graph\',\'adaptive\')==\'adaptive\':w=np.exp(-d*d/(np.maximum(d[:,-1,None],1e-8)*self.sigma[ix]+1e-12))\n        elif self.c.get(\'graph\')==\'binary\':w=np.ones_like(d)\n        else:w=np.exp(-d*d/max(self.band**2,1e-12))\n        exact=d[:,0]<1e-8\n        p=np.einsum(\'ij,ijk->ik\',w,self.F[ix])/np.maximum(w.sum(1)[:,None],1e-15)\n        p[exact]=self.F[ix[exact,0]]\n        return p\n\nclass Cluster:\n    def __init__(self,c):self.c=c\n    def fit(self,X,Y):\n        c=self.c;typ=c[\'cluster\'];n=c.get(\'clusters\',12)\n        if typ==\'kmeans\':model=KMeans(n_clusters=n,n_init=10,random_state=42)\n        elif typ==\'bisect\':model=BisectingKMeans(n_clusters=n,n_init=3,random_state=42)\n        elif typ==\'gmm\':model=GaussianMixture(n_components=n,covariance_type=c.get(\'cov\',\'diag\'),reg_covar=0.02,n_init=2,max_iter=200,random_state=42)\n        elif typ==\'agglomerative\':model=AgglomerativeClustering(n_clusters=n,linkage=c.get(\'linkage\',\'ward\'))\n        elif typ==\'spectral\':model=SpectralClustering(n_clusters=n,affinity=\'nearest_neighbors\',n_neighbors=c.get(\'k\',15),assign_labels=\'cluster_qr\',random_state=42)\n        elif typ==\'birch\':model=Birch(n_clusters=n,threshold=0.5)\n        elif typ==\'hdbscan\':model=HDBSCAN(min_cluster_size=c.get(\'min_cluster\',8),min_samples=c.get(\'min_samples\',3))\n        elif typ==\'dbscan\':\n            ds=NearestNeighbors(n_neighbors=6).fit(X).kneighbors()[0][:,-1]\n            model=DBSCAN(eps=float(np.quantile(ds,c.get(\'eps_q\',0.5))),min_samples=4)\n        else:raise ValueError(typ)\n        z=model.fit_predict(X);self.model=model;self.X=X;self.z=z\n        known=Y[:,0]>=0;self.prior=Y[known].mean(0)\n        ids=sorted(set(z)-{-1});self.ids=ids;self.centers=np.array([X[z==i].mean(0) for i in ids])\n        self.mapping={};smooth=c.get(\'smooth\',0.2)\n        for i in ids:\n            mask=known&(z==i);self.mapping[i]=(Y[mask].sum(0)+smooth*self.prior)/(mask.sum()+smooth)\n        self.fallback=KNeighborsRegressor(n_neighbors=min(5,known.sum()),weights=\'distance\').fit(X[known],Y[known])\n        self.known=known;self.Y=Y\n        if typ==\'gmm\':\n            R=model.predict_proba(X[known]);self.softmap=(R.T@Y[known]+smooth*self.prior)/(R.sum(0)[:,None]+smooth)\n        return self\n    def predict(self,Q):\n        typ=self.c[\'cluster\']\n        if typ==\'gmm\':return self.model.predict_proba(Q)@self.softmap\n        if hasattr(self.model,\'predict\'):z=self.model.predict(Q)\n        else:\n            # Inductive extension for algorithms without predict: nearest training member.\n            _,ix=NearestNeighbors(n_neighbors=1).fit(self.X).kneighbors(Q);z=self.z[ix[:,0]]\n        p=self.fallback.predict(Q)\n        for i in self.ids:\n            if np.sum(self.known&(self.z==i))>0:p[z==i]=self.mapping[i]\n        return p\n\ndef fit_supervised(X,Y,Q,c,weights=None):\n    typ=c[\'model\'];param=c.get(\'C\',1.);out=[]\n    if typ==\'knn\':return KNeighborsRegressor(n_neighbors=min(c.get(\'k\',5),len(X)),weights=c.get(\'weights\',\'distance\'),metric=c.get(\'metric\',\'euclidean\')).fit(X,Y).predict(Q)\n    if typ==\'ridge\':return np.clip(Ridge(alpha=c.get(\'alpha\',10)).fit(X,Y,sample_weight=weights).predict(Q),0,1)\n    if typ==\'krr\':return np.clip(KernelRidge(alpha=c.get(\'alpha\',.1),kernel=\'rbf\',gamma=c.get(\'gamma\',.003)).fit(X,Y,sample_weight=weights).predict(Q),0,1)\n    if typ==\'pls\':return np.clip(PLSRegression(n_components=c.get(\'components\',5)).fit(X,Y).predict(Q),0,1)\n    if typ in (\'softmax\',\'softsvc\'):\n        rows,classes=np.where(Y>0);ww=1/Y.sum(1)[rows]\n        if weights is not None:ww*=weights[rows]\n        if typ==\'softmax\':m=LogisticRegression(C=param,max_iter=2000)\n        else:m=SVC(C=param,gamma=c.get(\'gamma\',\'scale\'),probability=True,random_state=42)\n        m.fit(X[rows],classes,sample_weight=ww)\n        p=np.zeros((len(Q),Y.shape[1]));p[:,m.classes_]=m.predict_proba(Q);return p\n    if typ==\'nca\':\n        nca=NeighborhoodComponentsAnalysis(n_components=c.get(\'components\',5),max_iter=100,random_state=42)\n        xx=nca.fit_transform(X,Y@2**np.arange(5));qq=nca.transform(Q)\n        return KNeighborsRegressor(n_neighbors=c.get(\'k\',5),weights=\'distance\').fit(xx,Y).predict(qq)\n    if typ==\'chain\':\n        model=ClassifierChain(LogisticRegression(C=param,max_iter=1000),order=\'random\',random_state=c.get(\'seed\',42),cv=3,chain_method=\'predict_proba\')\n        return model.fit(X,Y).predict_proba(Q)\n    if typ==\'powerset\':\n        # Preserve multilabel patterns instead of imposing mutual exclusivity.\n        code=Y@2**np.arange(5)\n        if c.get(\'kernel\',\'rbf\')==\'linear\':model=LogisticRegression(C=param,max_iter=1000)\n        else:model=SVC(C=param,gamma=c.get(\'gamma\',\'scale\'),probability=True,random_state=42)\n        model.fit(X,code,sample_weight=weights);patterns=((model.classes_[:,None]>>np.arange(5))&1)\n        return model.predict_proba(Q)@patterns\n    for j in range(Y.shape[1]):\n        y=Y[:,j]\n        if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue\n        if typ==\'logistic\':m=LogisticRegression(C=param,max_iter=2000,class_weight=c.get(\'balanced\'),solver=\'lbfgs\')\n        elif typ==\'svc\':m=SVC(C=param,kernel=c.get(\'kernel\',\'rbf\'),gamma=c.get(\'gamma\',\'scale\'),degree=c.get(\'degree\',2),class_weight=c.get(\'balanced\'),probability=True,random_state=42)\n        elif typ==\'linear_svc\':m=CalibratedClassifierCV(LinearSVC(C=param,class_weight=c.get(\'balanced\'),max_iter=5000),cv=3)\n        elif typ==\'lda\':m=LinearDiscriminantAnalysis(solver=\'lsqr\',shrinkage=c.get(\'shrinkage\',\'auto\'))\n        elif typ==\'qda\':m=QuadraticDiscriminantAnalysis(reg_param=c.get(\'reg\',0.9))\n        elif typ==\'gnb\':m=GaussianNB(var_smoothing=c.get(\'smooth\',1e-2))\n        elif typ==\'gp\':m=GaussianProcessClassifier(kernel=ConstantKernel(c.get(\'amplitude\',1))*RBF(c.get(\'length\',10)),optimizer=None,random_state=42)\n        elif typ in (\'rf\',\'extra\'):\n            cls=RandomForestClassifier if typ==\'rf\' else ExtraTreesClassifier\n            m=cls(n_estimators=c.get(\'trees\',200),max_depth=c.get(\'depth\'),min_samples_leaf=c.get(\'leaf\',2),max_features=c.get(\'max_features\',\'sqrt\'),class_weight=c.get(\'balanced\'),random_state=42,n_jobs=1)\n        elif typ==\'hist\':m=HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=7,l2_regularization=5,random_state=42)\n        elif typ==\'catboost\':\n            from catboost import CatBoostClassifier\n            m=CatBoostClassifier(iterations=c.get(\'trees\',250),depth=c.get(\'depth\',3),learning_rate=0.04,l2_leaf_reg=5,verbose=False,thread_count=1,allow_writing_files=False,random_seed=42)\n        elif typ==\'xgb\':\n            from xgboost import XGBClassifier\n            m=XGBClassifier(n_estimators=c.get(\'trees\',150),max_depth=c.get(\'depth\',2),learning_rate=.04,reg_lambda=5,min_child_weight=2,subsample=.9,colsample_bytree=.8,n_jobs=1,random_state=42)\n        elif typ==\'lightgbm\':\n            from lightgbm import LGBMClassifier\n            m=LGBMClassifier(n_estimators=150,num_leaves=7,min_child_samples=10,learning_rate=.04,reg_lambda=5,verbosity=-1,n_jobs=1,random_state=42)\n        else:raise ValueError(typ)\n        if weights is not None and typ not in (\'lda\',\'qda\'):m.fit(X,y,sample_weight=weights)\n        else:m.fit(X,y)\n        out.append(m.predict_proba(Q)[:,1])\n    return np.array(out).T\n\ndef predict(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):\n    if c[\'model\']==\'ensemble\':\n        weights=np.asarray(c.get(\'weights\',np.ones(len(c[\'members\']))),float);weights/=weights.sum()\n        outputs=[predict(train_pos,query_frame,m,excluded_groups,return_unlabeled) for m in c[\'members\']]\n        if return_unlabeled:\n            return sum(w*x[0] for w,x in zip(weights,outputs)),sum(w*x[1] for w,x in zip(weights,outputs)),outputs[0][2]\n        return sum(w*x for w,x in zip(weights,outputs))\n    # Fit unsupervised transforms only on fold-training labels and legitimate masked rows.\n    actual=LAB[np.asarray(train_pos)]\n    u=UNLAB if excluded_groups is None else UNLAB[~np.isin(GROUP[UNLAB],excluded_groups)]\n    idx=np.r_[actual,u];n=len(actual)\n    A,B=features(idx,query_frame,c);y=Y[train_pos];ys=np.full((len(A),5),-1);ys[:n]=y\n    typ=c[\'model\']\n    if typ==\'semigmm\':\n        # Fractional membership handles multilabel gold rows; masked rows participate in EM.\n        # Labeled responsibilities remain fixed, and receive an explicit reliability weight.\n        z=y/np.maximum(y.sum(1)[:,None],1)\n        means=np.array([np.average(A[:n],axis=0,weights=z[:,j]+1e-8) for j in range(5)])\n        covariance=np.cov(A.T)+np.eye(A.shape[1])*.1\n        prior=(z.sum(0)+1)/(z.sum()+5)\n        labelweight=c.get(\'label_weight\',5.)\n        def posterior(Q):\n            ll=[]\n            for j in range(5):\n                cov=covs[j];factor=cho_factor(cov,lower=True);diff=Q-means[j]\n                ll.append(-.5*np.sum(diff*cho_solve(factor,diff.T).T,axis=1)-np.log(np.diag(factor[0])).sum()+np.log(prior[j]))\n            return softmax(np.array(ll).T,axis=1)\n        covs=np.array([covariance.copy() for _ in range(5)])\n        for iteration in range(80):\n            R=posterior(A);R[:n]=z;R[:n]*=labelweight\n            counts=R.sum(0)+1e-8;newmeans=R.T@A/counts[:,None]\n            delta=np.max(np.abs(newmeans-means));means=newmeans;prior=(counts+1)/(counts.sum()+5)\n            covlist=[]\n            for j in range(5):\n                diff=A-means[j];cov=(diff.T*R[:,j])@diff/counts[j]\n                shrink=c.get(\'shrink\',.8);cov=(1-shrink)*cov+shrink*covariance+np.eye(A.shape[1])*.05;covlist.append(cov)\n            if c.get(\'cov\',\'shared\')==\'shared\':\n                cov=np.average(covlist,axis=0,weights=counts);covs=np.array([cov]*5)\n            else:covs=np.array(covlist)\n            if delta<1e-5:break\n        p=posterior(B);pu=posterior(A[n:])\n    elif typ==\'laprls\':\n        # RBF kernel regression with graph-Laplacian regularization from masked data.\n        from sklearn.metrics.pairwise import rbf_kernel\n        gamma=c.get(\'gamma\',.003);K=rbf_kernel(A,gamma=gamma);KB=rbf_kernel(B,A,gamma=gamma)\n        nn=NearestNeighbors(n_neighbors=11).fit(A);dd,ii=nn.kneighbors(A);dd=dd[:,1:];ii=ii[:,1:]\n        W=sparse.csr_matrix((np.exp(-dd.ravel()**2/np.median(dd[:,-1])**2),(np.repeat(np.arange(len(A)),10),ii.ravel())),shape=(len(A),len(A)))\n        W=(W+W.T)*.5;degree=np.asarray(W.sum(1)).ravel();L=sparse.diags(degree)-W\n        # Normal equations in the representer coefficients; ridge makes the system stable.\n        mat=K[:n].T@K[:n]+c.get(\'alpha\',1)*K+c.get(\'mu\',1)*(K@(L@K))+np.eye(len(A))*1e-6\n        coef=cho_solve(cho_factor(mat,lower=True),K[:n].T@y)\n        p=np.clip(KB@coef,0,1);pu=np.clip(K[n:]@coef,0,1)\n    elif typ==\'cluster_features\':\n        # Unsupervised prototype dictionary from genuine training + masked rows only.\n        km=KMeans(n_clusters=c.get(\'clusters\',40),n_init=5,random_state=42).fit(A)\n        da=km.transform(A);db=km.transform(B)\n        band=np.median(np.min(da,axis=1))*c.get(\'band\',1.)\n        H=np.exp(-da**2/max(2*band**2,1e-8));J=np.exp(-db**2/max(2*band**2,1e-8))\n        sc=StandardScaler();H=sc.fit_transform(H);J=sc.transform(J)\n        mix=c.get(\'original_weight\',.3)\n        H=np.column_stack([H,A*mix]);J=np.column_stack([J,B*mix])\n        out=fit_supervised(H[:n],y,np.r_[J,H[n:]],{**c,\'model\':\'logistic\'})\n        p=out[:len(B)];pu=out[len(B):]\n    elif typ==\'metric_graph\':\n        # Label-informed metric fitted only to fold-training labels, followed by unsupervised graph inference.\n        base={**c,\'model\':c.get(\'metric_model\',\'logistic\')}\n        scores=fit_supervised(A[:n],y,np.r_[A,B],base)\n        scores=logit(np.clip(scores,.005,.995))\n        sc=StandardScaler();s=sc.fit_transform(scores[:len(A)]);q=sc.transform(scores[len(A):])\n        residual=c.get(\'residual\',.3)\n        norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))\n        U=np.column_stack([s,A*residual/max(norm,1e-8)])\n        V=np.column_stack([q,B*residual/max(norm,1e-8)])\n        model=Graph(c).fit(U,ys);p=model.predict(V);pu=model.F[n:]\n    elif typ==\'graph\':\n        model=Graph(c).fit(A,ys);p=model.predict(B);pu=model.F[n:]\n    elif typ==\'cluster\':\n        model=Cluster(c).fit(A,ys);p=model.predict(B);pu=model.predict(A[n:])\n    elif typ==\'pseudo\':\n        teacher=c[\'teacher\'];T,U=preprocess(DATA.iloc[idx][EMB],query_frame[EMB],teacher)\n        if teacher[\'model\']==\'graph\':m=Graph(teacher).fit(T,ys);pt=m.F[n:]\n        else:m=Cluster(teacher).fit(T,ys);pt=m.predict(T[n:])\n        student={**c,\'model\':c.get(\'student\',\'svc\')}\n        conf=c.get(\'confidence\',.9);w=c.get(\'pseudo_weight\',.2)\n        base_out=None\n        if c.get(\'consensus\',False):\n            base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)\n            base_u=base_out[len(B):]\n            agreement=(pt>=.5)==(base_u>=.5)\n            # Confidence must be supported by both independently constructed views.\n            pt=np.where(agreement,(pt+base_u)*.5,.5)\n        yp=(pt>=.5).astype(int)\n        rows=np.min(np.maximum(pt,1-pt),axis=1)>=conf\n        if c.get(\'labelwise\',False):\n            p=np.zeros((len(B),5));pu=np.zeros((len(u),5))\n            for j in range(5):\n                sel=np.maximum(pt[:,j],1-pt[:,j])>=conf\n                xx=np.r_[A[:n],A[n:][sel]];yy=np.r_[y[:,j],yp[sel,j]][:,None];ww=np.r_[np.ones(n),np.full(sel.sum(),w)]\n                out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p[:,j]=out[:len(B),0];pu[:,j]=out[len(B):,0]\n        else:\n            xx=np.r_[A[:n],A[n:][rows]];yy=np.r_[y,yp[rows]];ww=np.r_[np.ones(n),np.full(rows.sum(),w)]\n            out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p=out[:len(B)];pu=out[len(B):]\n        if c.get(\'base_mix\',0):\n            if base_out is None:base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)\n            mix=c[\'base_mix\'];p=(1-mix)*p+mix*base_out[:len(B)];pu=(1-mix)*pu+mix*base_out[len(B):]\n    else:\n        out=fit_supervised(A[:n],y,np.r_[B,A[n:]] if return_unlabeled else B,c)\n        p=out[:len(B)];pu=out[len(B):] if return_unlabeled else None\n    if return_unlabeled:return p,pu,u\n    return np.clip(p,0,1)\n\ndef evaluate(c,splits=None,seed=20261104,tag=\'screen\',force=False):\n    if splits is None:splits=make_splits(seed=seed)\n    cid=config_id(c);path=ROOT/\'experiments\'/f\'{tag}-{cid}-{seed}.npz\'\n    if path.exists() and not force:return dict(np.load(path,allow_pickle=True)[\'record\'].item())\n    started=time.time();p=np.full(Y.shape,np.nan);foldids=np.full(len(Y),-1)\n    try:\n        with threadpool_limits(limits=1):\n            for fold,(a,b) in enumerate(splits):\n                p[b]=predict(a,DATA.iloc[LAB[b]],c,excluded_groups=GROUP[LAB[b]])\n                foldids[b]=fold\n        use=foldids>=0;rule,score=choose_rule(Y[use],p[use]);base=metrics(Y[use],p[use]);tuned=metrics(Y[use],p[use],rule)\n        fs=[metrics(Y[b],p[b],rule)[\'sample_f1\'] for a,b in splits]\n        record={\'id\':cid,\'tag\':tag,\'seed\':seed,\'model\':c[\'model\'],\'config\':c,\'n_scored\':int(use.sum()),\'score\':score,\'rule\':rule,\'default_score\':base[\'sample_f1\'],\'fold_scores\':fs,\'fold_std\':float(np.std(fs)),**tuned,\'seconds\':round(time.time()-started,3),\'status\':\'ok\'}\n        np.savez_compressed(path,p=p,fold=foldids,record=record)\n    except Exception as e:\n        record={\'id\':cid,\'tag\':tag,\'seed\':seed,\'config\':c,\'status\':\'error\',\'error\':str(e),\'traceback\':traceback.format_exc(),\'seconds\':round(time.time()-started,3)}\n    with open(ROOT/\'experiments/trials.jsonl\',\'a\') as f:f.write(json.dumps(record)+\'\\n\')\n    print(json.dumps({k:record.get(k) for k in [\'id\',\'model\',\'score\',\'rule\',\'default_score\',\'seconds\',\'status\',\'error\']}),flush=True)\n    return record\n\ndef screen_configs():\n    out=[]\n    for scale in [\'standard\',\'global\']:\n        for C in [.01,.1,1,10,100]:out.append(dict(model=\'logistic\',C=C,scale=scale))\n        for C in [.1,1,10,100]:\n            for gamma in [.001,.003,.01,.03]:out.append(dict(model=\'svc\',C=C,gamma=gamma,scale=scale))\n        for k in [1,3,5,9,15]:\n            for weights in [\'uniform\',\'distance\']:out.append(dict(model=\'knn\',k=k,weights=weights,scale=scale))\n        for a in [1,10,100,1000]:out.append(dict(model=\'ridge\',alpha=a,scale=scale))\n        for C in [.01,.1,1,10]:out.append(dict(model=\'linear_svc\',C=C,scale=scale))\n    for typ in [\'logistic\',\'svc\']:\n        for C in [.1,1,10]:out.append(dict(model=typ,C=C,balanced=\'balanced\'))\n    for pc in [10,25,50,100]:\n        for white in [0,.5,1]:\n            for typ in [\'svc\',\'logistic\',\'knn\']:\n                out.append(dict(model=typ,pca=pc,white=white,C=10,k=3))\n    for typ in [\'logistic\',\'svc\',\'knn\']:\n        for scale in [\'standard\',\'global\',\'raw\']:out.append(dict(model=typ,scale=scale,norm=True,C=10,k=3,gamma=\'scale\'))\n    for typ in [\'lda\',\'qda\',\'gnb\',\'rf\',\'extra\',\'hist\',\'catboost\',\'xgb\',\'lightgbm\',\'chain\',\'powerset\']:out.append(dict(model=typ))\n    for typ in [\'rf\',\'extra\',\'catboost\',\'xgb\']:\n        for pc in [15,40]:out.append(dict(model=typ,pca=pc))\n    for typ in [\'kmeans\',\'bisect\',\'gmm\',\'agglomerative\',\'spectral\',\'birch\']:\n        for n in [5,12,25,50]:out.append(dict(model=\'cluster\',cluster=typ,clusters=n,pca=25))\n    for n in [5,12,25]:out.append(dict(model=\'cluster\',cluster=\'gmm\',clusters=n,pca=15,cov=\'full\'))\n    for mc in [5,10,20]:out.append(dict(model=\'cluster\',cluster=\'hdbscan\',min_cluster=mc,pca=25))\n    for q in [.25,.5,.75]:out.append(dict(model=\'cluster\',cluster=\'dbscan\',eps_q=q,pca=25))\n    for pc in [0,25,50]:\n        for k in [5,10,20,40]:\n            for alpha in [.8,.95,.99]:out.append(dict(model=\'graph\',pca=pc,k=k,alpha=alpha))\n    for k in [5,15,40]:\n        for alpha in [.95,.995]:out.append(dict(model=\'graph\',k=k,alpha=alpha,hard=True))\n    return out\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'--stage\',default=\'screen\');parser.add_argument(\'--configs\');parser.add_argument(\'--seed\',type=int,default=20261104);args=parser.parse_args()\n    configs=json.loads(Path(args.configs).read_text()) if args.configs else screen_configs()\n    save_json(ROOT/\'experiments\'/f\'{args.stage}-configs.json\',configs)\n    for i,c in enumerate(configs):\n        print(f\'RUN {i+1}/{len(configs)} {c}\',flush=True);evaluate(c,seed=args.seed,tag=args.stage)\n    records=[json.loads(s) for s in (ROOT/\'experiments/trials.jsonl\').read_text().splitlines()]\n    pd.DataFrame([{**r,\'config\':json.dumps(r[\'config\'],sort_keys=True)} for r in records]).to_csv(ROOT/\'experiments/trials.csv\',index=False)\n\nif __name__==\'__main__\':main()\n'
(replica / 'experiment.py').write_text(source)


29085

### experiment_v1.py

Complete source:

In [4]:
source = '"""Reproducible, train-only Part 1 development engine. See experiments/protocol.txt."""\nimport os\nfor var in [\'OMP_NUM_THREADS\',\'OPENBLAS_NUM_THREADS\',\'MKL_NUM_THREADS\',\'VECLIB_MAXIMUM_THREADS\',\'NUMEXPR_NUM_THREADS\']:\n    os.environ.setdefault(var,\'1\')\nimport argparse, hashlib, json, time, traceback, warnings, re, ast\nfrom pathlib import Path\nimport numpy as np\nimport pandas as pd\nfrom scipy import sparse\nfrom scipy.special import expit, softmax\nfrom scipy.special import logit\nfrom scipy.sparse.linalg import spsolve\nfrom sklearn.preprocessing import StandardScaler, normalize\nfrom sklearn.decomposition import PCA, TruncatedSVD\nfrom sklearn.feature_extraction.text import TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression, Ridge\nfrom sklearn.kernel_ridge import KernelRidge\nfrom sklearn.gaussian_process import GaussianProcessClassifier\nfrom sklearn.gaussian_process.kernels import RBF, ConstantKernel\nfrom sklearn.neighbors import NeighborhoodComponentsAnalysis\nfrom sklearn.svm import SVC, LinearSVC\nfrom sklearn.calibration import CalibratedClassifierCV\nfrom sklearn.neighbors import KNeighborsRegressor, NearestNeighbors\nfrom sklearn.naive_bayes import GaussianNB\nfrom sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis\nfrom sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier\nfrom sklearn.cluster import KMeans, BisectingKMeans, AgglomerativeClustering, DBSCAN, HDBSCAN, SpectralClustering, Birch\nfrom sklearn.mixture import GaussianMixture\nfrom sklearn.metrics import f1_score, pairwise_distances\nfrom sklearn.multioutput import ClassifierChain\nfrom sklearn.cross_decomposition import PLSRegression\nfrom iterstrat.ml_stratifiers import MultilabelStratifiedKFold\nfrom threadpoolctl import threadpool_limits\nwarnings.filterwarnings(\'ignore\',category=FutureWarning)\nROOT=Path(__file__).resolve().parent\nLABELS=[\'Safety\',\'Funding\',\'Efficacy\',\'Recruitment\',\'Other\']; EMB=[f\'emb_{i}\' for i in range(150)]\nDATA=pd.read_csv(ROOT/\'data/part1_train.csv\')\nLAB=np.flatnonzero(DATA[LABELS].ge(0).all(axis=1)); UNLAB=np.flatnonzero(DATA[LABELS].eq(-100).all(axis=1))\nY=DATA.iloc[LAB][LABELS].to_numpy(int)\nGROUP=pd.util.hash_pandas_object(DATA[EMB].round(10),index=False).to_numpy()\nassert len(LAB)+len(UNLAB)==len(DATA)\nCACHE={}\n\ndef config_id(c): return hashlib.sha256(json.dumps(c,sort_keys=True).encode()).hexdigest()[:14]\ndef save_json(path,obj): Path(path).write_text(json.dumps(obj,indent=2,default=lambda x:x.item() if isinstance(x,np.generic) else str(x)))\ndef make_splits(y=Y,groups=None,seed=20261104,n_splits=5):\n    if groups is None: groups=GROUP[LAB]\n    unique,inv=np.unique(groups,return_inverse=True)\n    gy=np.array([y[inv==i].max(axis=0) for i in range(len(unique))])\n    return [(np.flatnonzero(np.isin(inv,a)),np.flatnonzero(np.isin(inv,b))) for a,b in MultilabelStratifiedKFold(n_splits=n_splits,shuffle=True,random_state=seed).split(gy,gy)]\n\ndef decisions(p,rule=\'t0.5\'):\n    p=np.clip(np.asarray(p),0,1)\n    if rule==\'expected_f1\':\n        patterns=((np.arange(32)[:,None]>>np.arange(5))&1)\n        prob=np.prod(np.where(patterns[None,:,:],p[:,None,:],1-p[:,None,:]),axis=2)\n        choices=patterns[1:]\n        f1=2*(choices@patterns.T)/(choices.sum(1)[:,None]+patterns.sum(1)[None,:])\n        return choices[np.argmax(prob@f1.T,axis=1)]\n    if rule==\'argmax\':\n        out=np.zeros_like(p,dtype=int);out[np.arange(len(p)),p.argmax(1)]=1;return out\n    out=(p>=float(rule[1:])).astype(int)\n    empty=out.sum(1)==0\n    out[np.flatnonzero(empty),p[empty].argmax(1)]=1\n    return out\nRULES=[\'t0.25\',\'t0.35\',\'t0.45\',\'t0.5\',\'t0.55\',\'expected_f1\',\'argmax\']\ndef metrics(y,p,rule=\'t0.5\'):\n    pred=decisions(p,rule)\n    return {\'sample_f1\':float(f1_score(y,pred,average=\'samples\',zero_division=0)), \'macro_f1\':float(f1_score(y,pred,average=\'macro\',zero_division=0)),**{f\'f1_{l}\':float(v) for l,v in zip(LABELS,f1_score(y,pred,average=None,zero_division=0))}}\ndef choose_rule(y,p):\n    scores=[metrics(y,p,r)[\'sample_f1\'] for r in RULES]\n    # Prefer default threshold when differences are negligible.\n    best=max(scores); idx=min([i for i,s in enumerate(scores) if s>=best-1e-12],key=lambda i:abs(i-3))\n    return RULES[idx],best\n\ndef preprocess(A,B,c):\n    A=np.asarray(A,float);B=np.asarray(B,float)\n    mode=c.get(\'scale\',\'standard\')\n    if mode==\'standard\':\n        sc=StandardScaler();A=sc.fit_transform(A);B=sc.transform(B)\n    elif mode==\'global\':\n        mean=A.mean(0);std=np.sqrt(np.mean(np.var(A,axis=0)));A=(A-mean)/std;B=(B-mean)/std\n    elif mode==\'raw\':\n        # Global scale without centering preserves cosine and original relative geometry.\n        std=np.sqrt(np.mean(np.var(A,axis=0)));A=A/std;B=B/std\n    n=c.get(\'pca\',0)\n    if n:\n        pc=PCA(n_components=min(n,A.shape[1],A.shape[0]-1),svd_solver=\'full\');A=pc.fit_transform(A);B=pc.transform(B)\n        power=c.get(\'white\',0)\n        if power:\n            scale=np.maximum(pc.explained_variance_,1e-8)**(power/2);A/=scale;B/=scale\n    if c.get(\'norm\',False):A=normalize(A);B=normalize(B)\n    return A,B\n\ndef chemical(frame,kind):\n    from rdkit import Chem, RDLogger\n    from rdkit.Chem import Descriptors, rdFingerprintGenerator\n    RDLogger.DisableLog(\'rdApp.*\')\n    gen=rdFingerprintGenerator.GetMorganGenerator(radius=2,fpSize=1024)\n    desc=[Descriptors.MolWt,Descriptors.MolLogP,Descriptors.TPSA,Descriptors.NumHDonors,Descriptors.NumHAcceptors,Descriptors.NumRotatableBonds,Descriptors.RingCount,Descriptors.FractionCSP3,Descriptors.HeavyAtomCount]\n    out=[]\n    for field in frame.drug_smiles.fillna(\'\'):\n        mols=[m for s in field.split(\';\') if s.strip() and (m:=Chem.MolFromSmiles(s.strip())) is not None]\n        if kind==\'fingerprint\':v=np.max([gen.GetFingerprintAsNumPy(m) for m in mols],axis=0) if mols else np.zeros(1024)\n        else:\n            z=np.array([[fn(m) for fn in desc] for m in mols])\n            v=np.r_[len(mols),z.mean(0),z.max(0)] if mols else np.zeros(19)\n        out.append(v)\n    return np.array(out,float)\n\ndef block_features(A,B,block):\n    if block in (\'fingerprint\',\'descriptors\'):\n        a=chemical(A,block);b=chemical(B,block)\n    elif block==\'year\':a=A[[\'start_year\']].to_numpy();b=B[[\'start_year\']].to_numpy()\n    elif block==\'missingness\':\n        cols=[\'criteria\',\'diseases_cleaned\',\'drugs\',\'icdcodes\',\'matched_drugs\',\'drug_targets\',\'drug_actions\',\'drug_pathways\',\'drug_smiles\']\n        a=A[cols].isna().astype(float).to_numpy();b=B[cols].isna().astype(float).to_numpy()\n    elif block==\'counts\':\n        cols=[\'criteria\',\'diseases_cleaned\',\'drugs\',\'icdcodes\',\'matched_drugs\',\'drug_targets\',\'drug_actions\',\'drug_pathways\',\'drug_smiles\']\n        def counts(D):return np.column_stack([np.log1p(D[c].fillna(\'\').str.len()) for c in cols]+[D[c].fillna(\'\').str.count(\';\')+D[c].notna() for c in cols[4:]])\n        a=counts(A);b=counts(B)\n    elif block.startswith(\'pathway_level\'):\n        catalog=pd.read_csv(ROOT/\'data/pathway_catalog.csv\');levels=dict(zip(catalog.reactome_id,catalog.pathway_level))\n        if block==\'pathway_levels\':\n            def calc(D):return np.array([[sum(levels.get(s.strip(),-1)==i for s in str(v).split(\';\')) for i in range(10)] for v in D.drug_pathways.fillna(\'\')],float)\n            a=np.log1p(calc(A));b=np.log1p(calc(B))\n        else:\n            lev=int(block[-1]); allowed=set(catalog.loc[catalog.pathway_level<=lev,\'reactome_id\'])\n            def clean(D):return D.drug_pathways.fillna(\'\').map(lambda x:\' \'.join(t.strip() for t in x.split(\';\') if t.strip() in allowed))\n            v=TfidfVectorizer(token_pattern=r\'[^\\s]+\',min_df=2,max_features=1000);a=v.fit_transform(clean(A)).toarray();b=v.transform(clean(B)).toarray();return a,b\n    else:\n        col=block.replace(\'_char\',\'\');char=block.endswith(\'_char\')\n        v=TfidfVectorizer(lowercase=True,analyzer=\'char_wb\' if char else \'word\',ngram_range=(3,5) if char else (1,2),min_df=2,max_df=0.99,max_features=3000,sublinear_tf=True,token_pattern=r\'(?u)\\b[\\w-]+\\b\')\n        a=v.fit_transform(A[col].fillna(\'\'));b=v.transform(B[col].fillna(\'\'))\n        # Keep exact sparse-text geometry; at this dataset size dense blocks fit in memory.\n        return a.toarray(),b.toarray()\n    if block!=\'fingerprint\':\n        sc=StandardScaler();a=sc.fit_transform(a);b=sc.transform(b)\n    a=np.nan_to_num(a);b=np.nan_to_num(b)\n    norm=np.sqrt(np.mean(np.sum(a*a,axis=1)))\n    return a/max(norm,1e-8),b/max(norm,1e-8)\n\ndef features(train_idx,query_frame,c):\n    fitframe=DATA.iloc[train_idx]\n    A,B=preprocess(fitframe[EMB],query_frame[EMB],c)\n    blocks=c.get(\'features\',[])\n    if blocks:\n        base_norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))\n        weight=c.get(\'feature_weight\',0.3)\n        for block in blocks:\n            a,b=block_features(fitframe,query_frame,block)\n            A=np.column_stack([A,a*base_norm*weight]);B=np.column_stack([B,b*base_norm*weight])\n    return A,B\n\nclass Graph:\n    """Normalized graph diffusion of all five binary labels; independent negative mass implicit."""\n    def __init__(self,c):self.c=c\n    def fit(self,X,Y):\n        self.X=X; n=len(X);c=self.c;k=min(c.get(\'k\',15),n-1);self.k=k\n        self.nn=NearestNeighbors(n_neighbors=k+1).fit(X);dist,ix=self.nn.kneighbors(X)\n        dist=dist[:,1:];ix=ix[:,1:];self.sigma=np.maximum(dist[:,-1],1e-8)\n        self.band=float(np.median(dist[:,-1]))*c.get(\'band\',1.)\n        if c.get(\'graph\',\'adaptive\')==\'adaptive\':w=np.exp(-dist**2/(self.sigma[:,None]*self.sigma[ix]+1e-12))\n        elif c.get(\'graph\')==\'binary\':w=np.ones_like(dist)\n        else:w=np.exp(-dist**2/max(self.band**2,1e-12))\n        W=sparse.csr_matrix((w.ravel(),(np.repeat(np.arange(n),k),ix.ravel())),shape=(n,n))\n        W=(W+W.T)*0.5\n        degree=np.maximum(np.asarray(W.sum(1)).ravel(),1e-12)\n        known=Y[:,0]>=0\n        if c.get(\'hard\',False):\n            P=sparse.diags(1/degree)@W;u=np.flatnonzero(~known);l=np.flatnonzero(known)\n            alpha=c.get(\'alpha\',0.99)\n            U=spsolve(sparse.eye(len(u),format=\'csc\')-alpha*P[u][:,u].tocsc(),alpha*P[u][:,l]@Y[l]+(1-alpha)*Y[l].mean(0))\n            self.F=Y.copy().astype(float);self.F[u]=U\n        else:\n            S=sparse.diags(degree**-0.5)@W@sparse.diags(degree**-0.5)\n            Z=np.zeros((n,6));Z[known,:5]=Y[known];Z[known,5]=1\n            sol=spsolve(sparse.eye(n,format=\'csc\')-c.get(\'alpha\',0.95)*S.tocsc(),Z)\n            self.F=np.divide(sol[:,:5],sol[:,5,None],out=np.tile(Y[known].mean(0),(n,1)),where=sol[:,5,None]>1e-15)\n        self.F=np.clip(self.F,0,1);return self\n    def predict(self,Q):\n        d,ix=self.nn.kneighbors(Q,n_neighbors=self.k)\n        if self.c.get(\'graph\',\'adaptive\')==\'adaptive\':w=np.exp(-d*d/(np.maximum(d[:,-1,None],1e-8)*self.sigma[ix]+1e-12))\n        elif self.c.get(\'graph\')==\'binary\':w=np.ones_like(d)\n        else:w=np.exp(-d*d/max(self.band**2,1e-12))\n        exact=d[:,0]<1e-8\n        p=np.einsum(\'ij,ijk->ik\',w,self.F[ix])/np.maximum(w.sum(1)[:,None],1e-15)\n        p[exact]=self.F[ix[exact,0]]\n        return p\n\nclass Cluster:\n    def __init__(self,c):self.c=c\n    def fit(self,X,Y):\n        c=self.c;typ=c[\'cluster\'];n=c.get(\'clusters\',12)\n        if typ==\'kmeans\':model=KMeans(n_clusters=n,n_init=10,random_state=42)\n        elif typ==\'bisect\':model=BisectingKMeans(n_clusters=n,n_init=3,random_state=42)\n        elif typ==\'gmm\':model=GaussianMixture(n_components=n,covariance_type=c.get(\'cov\',\'diag\'),reg_covar=0.02,n_init=2,max_iter=200,random_state=42)\n        elif typ==\'agglomerative\':model=AgglomerativeClustering(n_clusters=n,linkage=c.get(\'linkage\',\'ward\'))\n        elif typ==\'spectral\':model=SpectralClustering(n_clusters=n,affinity=\'nearest_neighbors\',n_neighbors=c.get(\'k\',15),assign_labels=\'cluster_qr\',random_state=42)\n        elif typ==\'birch\':model=Birch(n_clusters=n,threshold=0.5)\n        elif typ==\'hdbscan\':model=HDBSCAN(min_cluster_size=c.get(\'min_cluster\',8),min_samples=c.get(\'min_samples\',3))\n        elif typ==\'dbscan\':\n            ds=NearestNeighbors(n_neighbors=6).fit(X).kneighbors()[0][:,-1]\n            model=DBSCAN(eps=float(np.quantile(ds,c.get(\'eps_q\',0.5))),min_samples=4)\n        else:raise ValueError(typ)\n        z=model.fit_predict(X);self.model=model;self.X=X;self.z=z\n        known=Y[:,0]>=0;self.prior=Y[known].mean(0)\n        ids=sorted(set(z)-{-1});self.ids=ids;self.centers=np.array([X[z==i].mean(0) for i in ids])\n        self.mapping={};smooth=c.get(\'smooth\',0.2)\n        for i in ids:\n            mask=known&(z==i);self.mapping[i]=(Y[mask].sum(0)+smooth*self.prior)/(mask.sum()+smooth)\n        self.fallback=KNeighborsRegressor(n_neighbors=min(5,known.sum()),weights=\'distance\').fit(X[known],Y[known])\n        self.known=known;self.Y=Y\n        if typ==\'gmm\':\n            R=model.predict_proba(X[known]);self.softmap=(R.T@Y[known]+smooth*self.prior)/(R.sum(0)[:,None]+smooth)\n        return self\n    def predict(self,Q):\n        typ=self.c[\'cluster\']\n        if typ==\'gmm\':return self.model.predict_proba(Q)@self.softmap\n        if hasattr(self.model,\'predict\'):z=self.model.predict(Q)\n        else:\n            # Inductive extension for algorithms without predict: nearest training member.\n            _,ix=NearestNeighbors(n_neighbors=1).fit(self.X).kneighbors(Q);z=self.z[ix[:,0]]\n        p=self.fallback.predict(Q)\n        for i in self.ids:\n            if np.sum(self.known&(self.z==i))>0:p[z==i]=self.mapping[i]\n        return p\n\ndef fit_supervised(X,Y,Q,c,weights=None):\n    typ=c[\'model\'];param=c.get(\'C\',1.);out=[]\n    if typ==\'knn\':return KNeighborsRegressor(n_neighbors=min(c.get(\'k\',5),len(X)),weights=c.get(\'weights\',\'distance\'),metric=c.get(\'metric\',\'euclidean\')).fit(X,Y).predict(Q)\n    if typ==\'ridge\':return np.clip(Ridge(alpha=c.get(\'alpha\',10)).fit(X,Y,sample_weight=weights).predict(Q),0,1)\n    if typ==\'krr\':return np.clip(KernelRidge(alpha=c.get(\'alpha\',.1),kernel=\'rbf\',gamma=c.get(\'gamma\',.003)).fit(X,Y,sample_weight=weights).predict(Q),0,1)\n    if typ==\'pls\':return np.clip(PLSRegression(n_components=c.get(\'components\',5)).fit(X,Y).predict(Q),0,1)\n    if typ==\'nca\':\n        nca=NeighborhoodComponentsAnalysis(n_components=c.get(\'components\',5),max_iter=100,random_state=42)\n        xx=nca.fit_transform(X,Y@2**np.arange(5));qq=nca.transform(Q)\n        return KNeighborsRegressor(n_neighbors=c.get(\'k\',5),weights=\'distance\').fit(xx,Y).predict(qq)\n    if typ==\'chain\':\n        model=ClassifierChain(LogisticRegression(C=param,max_iter=1000),order=\'random\',random_state=c.get(\'seed\',42),cv=3,chain_method=\'predict_proba\')\n        return model.fit(X,Y).predict_proba(Q)\n    if typ==\'powerset\':\n        # Preserve multilabel patterns instead of imposing mutual exclusivity.\n        code=Y@2**np.arange(5)\n        if c.get(\'kernel\',\'rbf\')==\'linear\':model=LogisticRegression(C=param,max_iter=1000)\n        else:model=SVC(C=param,gamma=c.get(\'gamma\',\'scale\'),probability=True,random_state=42)\n        model.fit(X,code,sample_weight=weights);patterns=((model.classes_[:,None]>>np.arange(5))&1)\n        return model.predict_proba(Q)@patterns\n    for j in range(Y.shape[1]):\n        y=Y[:,j]\n        if len(np.unique(y))<2:out.append(np.full(len(Q),y[0]));continue\n        if typ==\'logistic\':m=LogisticRegression(C=param,max_iter=2000,class_weight=c.get(\'balanced\'),solver=\'lbfgs\')\n        elif typ==\'svc\':m=SVC(C=param,kernel=c.get(\'kernel\',\'rbf\'),gamma=c.get(\'gamma\',\'scale\'),degree=c.get(\'degree\',2),class_weight=c.get(\'balanced\'),probability=True,random_state=42)\n        elif typ==\'linear_svc\':m=CalibratedClassifierCV(LinearSVC(C=param,class_weight=c.get(\'balanced\'),max_iter=5000),cv=3)\n        elif typ==\'lda\':m=LinearDiscriminantAnalysis(solver=\'lsqr\',shrinkage=c.get(\'shrinkage\',\'auto\'))\n        elif typ==\'qda\':m=QuadraticDiscriminantAnalysis(reg_param=c.get(\'reg\',0.9))\n        elif typ==\'gnb\':m=GaussianNB(var_smoothing=c.get(\'smooth\',1e-2))\n        elif typ==\'gp\':m=GaussianProcessClassifier(kernel=ConstantKernel(c.get(\'amplitude\',1))*RBF(c.get(\'length\',10)),optimizer=None,random_state=42)\n        elif typ in (\'rf\',\'extra\'):\n            cls=RandomForestClassifier if typ==\'rf\' else ExtraTreesClassifier\n            m=cls(n_estimators=c.get(\'trees\',200),max_depth=c.get(\'depth\'),min_samples_leaf=c.get(\'leaf\',2),max_features=c.get(\'max_features\',\'sqrt\'),class_weight=c.get(\'balanced\'),random_state=42,n_jobs=1)\n        elif typ==\'hist\':m=HistGradientBoostingClassifier(max_iter=150,max_leaf_nodes=7,l2_regularization=5,random_state=42)\n        elif typ==\'catboost\':\n            from catboost import CatBoostClassifier\n            m=CatBoostClassifier(iterations=c.get(\'trees\',250),depth=c.get(\'depth\',3),learning_rate=0.04,l2_leaf_reg=5,verbose=False,thread_count=1,allow_writing_files=False,random_seed=42)\n        elif typ==\'xgb\':\n            from xgboost import XGBClassifier\n            m=XGBClassifier(n_estimators=c.get(\'trees\',150),max_depth=c.get(\'depth\',2),learning_rate=.04,reg_lambda=5,min_child_weight=2,subsample=.9,colsample_bytree=.8,n_jobs=1,random_state=42)\n        elif typ==\'lightgbm\':\n            from lightgbm import LGBMClassifier\n            m=LGBMClassifier(n_estimators=150,num_leaves=7,min_child_samples=10,learning_rate=.04,reg_lambda=5,verbosity=-1,n_jobs=1,random_state=42)\n        else:raise ValueError(typ)\n        if weights is not None and typ not in (\'lda\',\'qda\'):m.fit(X,y,sample_weight=weights)\n        else:m.fit(X,y)\n        out.append(m.predict_proba(Q)[:,1])\n    return np.array(out).T\n\ndef predict(train_pos,query_frame,c,excluded_groups=None,return_unlabeled=False):\n    if c[\'model\']==\'ensemble\':\n        weights=np.asarray(c.get(\'weights\',np.ones(len(c[\'members\']))),float);weights/=weights.sum()\n        outputs=[predict(train_pos,query_frame,m,excluded_groups,return_unlabeled) for m in c[\'members\']]\n        if return_unlabeled:\n            return sum(w*x[0] for w,x in zip(weights,outputs)),sum(w*x[1] for w,x in zip(weights,outputs)),outputs[0][2]\n        return sum(w*x for w,x in zip(weights,outputs))\n    # Fit unsupervised transforms only on fold-training labels and legitimate masked rows.\n    actual=LAB[np.asarray(train_pos)]\n    u=UNLAB if excluded_groups is None else UNLAB[~np.isin(GROUP[UNLAB],excluded_groups)]\n    idx=np.r_[actual,u];n=len(actual)\n    A,B=features(idx,query_frame,c);y=Y[train_pos];ys=np.full((len(A),5),-1);ys[:n]=y\n    typ=c[\'model\']\n    if typ==\'cluster_features\':\n        # Unsupervised prototype dictionary from genuine training + masked rows only.\n        km=KMeans(n_clusters=c.get(\'clusters\',40),n_init=5,random_state=42).fit(A)\n        da=km.transform(A);db=km.transform(B)\n        band=np.median(np.min(da,axis=1))*c.get(\'band\',1.)\n        H=np.exp(-da**2/max(2*band**2,1e-8));J=np.exp(-db**2/max(2*band**2,1e-8))\n        sc=StandardScaler();H=sc.fit_transform(H);J=sc.transform(J)\n        mix=c.get(\'original_weight\',.3)\n        H=np.column_stack([H,A*mix]);J=np.column_stack([J,B*mix])\n        out=fit_supervised(H[:n],y,np.r_[J,H[n:]],{**c,\'model\':\'logistic\'})\n        p=out[:len(B)];pu=out[len(B):]\n    elif typ==\'metric_graph\':\n        # Label-informed metric fitted only to fold-training labels, followed by unsupervised graph inference.\n        base={**c,\'model\':c.get(\'metric_model\',\'logistic\')}\n        scores=fit_supervised(A[:n],y,np.r_[A,B],base)\n        scores=logit(np.clip(scores,.005,.995))\n        sc=StandardScaler();s=sc.fit_transform(scores[:len(A)]);q=sc.transform(scores[len(A):])\n        residual=c.get(\'residual\',.3)\n        norm=np.sqrt(np.mean(np.sum(A*A,axis=1)))\n        U=np.column_stack([s,A*residual/max(norm,1e-8)])\n        V=np.column_stack([q,B*residual/max(norm,1e-8)])\n        model=Graph(c).fit(U,ys);p=model.predict(V);pu=model.F[n:]\n    elif typ==\'graph\':\n        model=Graph(c).fit(A,ys);p=model.predict(B);pu=model.F[n:]\n    elif typ==\'cluster\':\n        model=Cluster(c).fit(A,ys);p=model.predict(B);pu=model.predict(A[n:])\n    elif typ==\'pseudo\':\n        teacher=c[\'teacher\'];T,U=preprocess(DATA.iloc[idx][EMB],query_frame[EMB],teacher)\n        if teacher[\'model\']==\'graph\':m=Graph(teacher).fit(T,ys);pt=m.F[n:]\n        else:m=Cluster(teacher).fit(T,ys);pt=m.predict(T[n:])\n        student={**c,\'model\':c.get(\'student\',\'svc\')}\n        conf=c.get(\'confidence\',.9);w=c.get(\'pseudo_weight\',.2)\n        base_out=None\n        if c.get(\'consensus\',False):\n            base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)\n            base_u=base_out[len(B):]\n            agreement=(pt>=.5)==(base_u>=.5)\n            # Confidence must be supported by both independently constructed views.\n            pt=np.where(agreement,(pt+base_u)*.5,.5)\n        yp=(pt>=.5).astype(int)\n        rows=np.min(np.maximum(pt,1-pt),axis=1)>=conf\n        if c.get(\'labelwise\',False):\n            p=np.zeros((len(B),5));pu=np.zeros((len(u),5))\n            for j in range(5):\n                sel=np.maximum(pt[:,j],1-pt[:,j])>=conf\n                xx=np.r_[A[:n],A[n:][sel]];yy=np.r_[y[:,j],yp[sel,j]][:,None];ww=np.r_[np.ones(n),np.full(sel.sum(),w)]\n                out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p[:,j]=out[:len(B),0];pu[:,j]=out[len(B):,0]\n        else:\n            xx=np.r_[A[:n],A[n:][rows]];yy=np.r_[y,yp[rows]];ww=np.r_[np.ones(n),np.full(rows.sum(),w)]\n            out=fit_supervised(xx,yy,np.r_[B,A[n:]],student,ww);p=out[:len(B)];pu=out[len(B):]\n        if c.get(\'base_mix\',0):\n            if base_out is None:base_out=fit_supervised(A[:n],y,np.r_[B,A[n:]],student)\n            mix=c[\'base_mix\'];p=(1-mix)*p+mix*base_out[:len(B)];pu=(1-mix)*pu+mix*base_out[len(B):]\n    else:\n        out=fit_supervised(A[:n],y,np.r_[B,A[n:]] if return_unlabeled else B,c)\n        p=out[:len(B)];pu=out[len(B):] if return_unlabeled else None\n    if return_unlabeled:return p,pu,u\n    return np.clip(p,0,1)\n\ndef evaluate(c,splits=None,seed=20261104,tag=\'screen\',force=False):\n    if splits is None:splits=make_splits(seed=seed)\n    cid=config_id(c);path=ROOT/\'experiments\'/f\'{tag}-{cid}-{seed}.npz\'\n    if path.exists() and not force:return dict(np.load(path,allow_pickle=True)[\'record\'].item())\n    started=time.time();p=np.full(Y.shape,np.nan);foldids=np.full(len(Y),-1)\n    try:\n        with threadpool_limits(limits=1):\n            for fold,(a,b) in enumerate(splits):\n                p[b]=predict(a,DATA.iloc[LAB[b]],c,excluded_groups=GROUP[LAB[b]])\n                foldids[b]=fold\n        use=foldids>=0;rule,score=choose_rule(Y[use],p[use]);base=metrics(Y[use],p[use]);tuned=metrics(Y[use],p[use],rule)\n        fs=[metrics(Y[b],p[b],rule)[\'sample_f1\'] for a,b in splits]\n        record={\'id\':cid,\'tag\':tag,\'seed\':seed,\'model\':c[\'model\'],\'config\':c,\'n_scored\':int(use.sum()),\'score\':score,\'rule\':rule,\'default_score\':base[\'sample_f1\'],\'fold_scores\':fs,\'fold_std\':float(np.std(fs)),**tuned,\'seconds\':round(time.time()-started,3),\'status\':\'ok\'}\n        np.savez_compressed(path,p=p,fold=foldids,record=record)\n    except Exception as e:\n        record={\'id\':cid,\'tag\':tag,\'seed\':seed,\'config\':c,\'status\':\'error\',\'error\':str(e),\'traceback\':traceback.format_exc(),\'seconds\':round(time.time()-started,3)}\n    with open(ROOT/\'experiments/trials.jsonl\',\'a\') as f:f.write(json.dumps(record)+\'\\n\')\n    print(json.dumps({k:record.get(k) for k in [\'id\',\'model\',\'score\',\'rule\',\'default_score\',\'seconds\',\'status\',\'error\']}),flush=True)\n    return record\n\ndef screen_configs():\n    out=[]\n    for scale in [\'standard\',\'global\']:\n        for C in [.01,.1,1,10,100]:out.append(dict(model=\'logistic\',C=C,scale=scale))\n        for C in [.1,1,10,100]:\n            for gamma in [.001,.003,.01,.03]:out.append(dict(model=\'svc\',C=C,gamma=gamma,scale=scale))\n        for k in [1,3,5,9,15]:\n            for weights in [\'uniform\',\'distance\']:out.append(dict(model=\'knn\',k=k,weights=weights,scale=scale))\n        for a in [1,10,100,1000]:out.append(dict(model=\'ridge\',alpha=a,scale=scale))\n        for C in [.01,.1,1,10]:out.append(dict(model=\'linear_svc\',C=C,scale=scale))\n    for typ in [\'logistic\',\'svc\']:\n        for C in [.1,1,10]:out.append(dict(model=typ,C=C,balanced=\'balanced\'))\n    for pc in [10,25,50,100]:\n        for white in [0,.5,1]:\n            for typ in [\'svc\',\'logistic\',\'knn\']:\n                out.append(dict(model=typ,pca=pc,white=white,C=10,k=3))\n    for typ in [\'logistic\',\'svc\',\'knn\']:\n        for scale in [\'standard\',\'global\',\'raw\']:out.append(dict(model=typ,scale=scale,norm=True,C=10,k=3,gamma=\'scale\'))\n    for typ in [\'lda\',\'qda\',\'gnb\',\'rf\',\'extra\',\'hist\',\'catboost\',\'xgb\',\'lightgbm\',\'chain\',\'powerset\']:out.append(dict(model=typ))\n    for typ in [\'rf\',\'extra\',\'catboost\',\'xgb\']:\n        for pc in [15,40]:out.append(dict(model=typ,pca=pc))\n    for typ in [\'kmeans\',\'bisect\',\'gmm\',\'agglomerative\',\'spectral\',\'birch\']:\n        for n in [5,12,25,50]:out.append(dict(model=\'cluster\',cluster=typ,clusters=n,pca=25))\n    for n in [5,12,25]:out.append(dict(model=\'cluster\',cluster=\'gmm\',clusters=n,pca=15,cov=\'full\'))\n    for mc in [5,10,20]:out.append(dict(model=\'cluster\',cluster=\'hdbscan\',min_cluster=mc,pca=25))\n    for q in [.25,.5,.75]:out.append(dict(model=\'cluster\',cluster=\'dbscan\',eps_q=q,pca=25))\n    for pc in [0,25,50]:\n        for k in [5,10,20,40]:\n            for alpha in [.8,.95,.99]:out.append(dict(model=\'graph\',pca=pc,k=k,alpha=alpha))\n    for k in [5,15,40]:\n        for alpha in [.95,.995]:out.append(dict(model=\'graph\',k=k,alpha=alpha,hard=True))\n    return out\n\ndef main():\n    parser=argparse.ArgumentParser();parser.add_argument(\'--stage\',default=\'screen\');parser.add_argument(\'--configs\');parser.add_argument(\'--seed\',type=int,default=20261104);args=parser.parse_args()\n    configs=json.loads(Path(args.configs).read_text()) if args.configs else screen_configs()\n    save_json(ROOT/\'experiments\'/f\'{args.stage}-configs.json\',configs)\n    for i,c in enumerate(configs):\n        print(f\'RUN {i+1}/{len(configs)} {c}\',flush=True);evaluate(c,seed=args.seed,tag=args.stage)\n    records=[json.loads(s) for s in (ROOT/\'experiments/trials.jsonl\').read_text().splitlines()]\n    pd.DataFrame([{**r,\'config\':json.dumps(r[\'config\'],sort_keys=True)} for r in records]).to_csv(ROOT/\'experiments/trials.csv\',index=False)\n\nif __name__==\'__main__\':main()\n'
(replica / 'experiment_v1.py').write_text(source)


26073

### finalize.py

Complete source:

In [5]:
source = '"""Fit frozen candidates, evaluate validation once, and generate Kaggle artifacts."""\nimport experiment_v1 as e\nimport pandas as pd,numpy as np,json,datetime,hashlib\nfrom pathlib import Path\nfrom threadpoolctl import threadpool_limits\nfrozen=json.loads(Path(\'experiments/frozen-shortlist.json\').read_text())\nassert hashlib.sha256(Path(\'experiment_v1.py\').read_bytes()).hexdigest()==frozen[\'engine_sha256\']\n# Read only predictor columns until all candidate predictions are committed to disk.\nval=pd.read_csv(\'data/part1_val.csv\',usecols=lambda c:c not in e.LABELS)\ntest=pd.read_csv(\'data/part1_test.csv\');unlab=e.DATA.iloc[e.UNLAB].drop(columns=e.LABELS)\nquery=pd.concat([val,test,unlab],ignore_index=True)\nseeds=[20261104,20261105,20261106];member_predictions={}\nwith threadpool_limits(limits=1):\n for member in frozen[\'members\']:\n  cid=e.config_id(member);out=Path(\'predictions\')/f\'locked-member-{cid}.npz\'\n  if out.exists():\n   z=np.load(out);assert np.array_equal(z[\'ids\'],query.ID.to_numpy(str));member_predictions[cid]=z[\'p\'];continue\n  preds=[]\n  for seed in seeds:\n   for fold,(a,b) in enumerate(e.make_splits(seed=seed)):\n    print(\'FIT\',cid,seed,fold,flush=True)\n    preds.append(e.predict(a,query,member))\n  p=np.mean(preds,axis=0);np.savez_compressed(out,p=p,ids=query.ID.to_numpy(str),seed_predictions=np.array(preds));member_predictions[cid]=p\n for candidate in frozen[\'shortlist\']:\n  c=candidate[\'config\'];weights=np.array(c[\'weights\'],float);weights/=weights.sum()\n  p=sum(w*member_predictions[e.config_id(m)] for w,m in zip(weights,c[\'members\']))\n  np.savez_compressed(\'predictions/locked-\'+candidate[\'name\']+\'.npz\',p=p,ids=query.ID.to_numpy(str))\n# Now and only now use validation labels for the frozen comparison.\nresult_path=Path(\'experiments/validation-once.json\')\nif result_path.exists():raise SystemExit(\'Validation comparison already exists. Refusing to run another comparison.\')\ny=pd.read_csv(\'data/part1_val.csv\',usecols=e.LABELS)[e.LABELS].to_numpy(int)\nrecords=[]\nfor candidate in frozen[\'shortlist\']:\n p=np.load(\'predictions/locked-\'+candidate[\'name\']+\'.npz\')[\'p\'][:len(val)]\n r={**candidate,**e.metrics(y,p,candidate[\'rule\'])};records.append(r);print(\'VALIDATION\',candidate[\'name\'],r[\'sample_f1\'],r[\'macro_f1\'],flush=True)\nmaximum=max(r[\'sample_f1\'] for r in records)\npriority={\'fingerprint_ssl\':0,\'embedding_ssl\':1,\'balanced_ensemble\':2}\nwinner=min([r for r in records if r[\'sample_f1\']>=maximum-.005],key=lambda r:priority[r[\'name\']])\np=np.load(\'predictions/locked-\'+winner[\'name\']+\'.npz\')[\'p\'];hard=e.decisions(p,winner[\'rule\'])\n# Cluster bootstrap by exact embedding group, keeping duplicate reasons together.\nvg=pd.util.hash_pandas_object(val[e.EMB].round(10),index=False).to_numpy();groups=np.unique(vg)\nrow_f1=2*(y*hard[:len(val)]).sum(1)/(y.sum(1)+hard[:len(val)].sum(1))\nrng=np.random.default_rng(8102026);boot=[]\nfor _ in range(10000):\n sampled=rng.choice(groups,len(groups),replace=True);idx=np.concatenate([np.flatnonzero(vg==g) for g in sampled]);boot.append(float(row_f1[idx].mean()))\nci=np.quantile(boot,[.025,.975]).tolist()\nresult=dict(evaluated_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),validation_rows=len(val),candidates=records,selected=winner[\'name\'],selected_sample_f1=winner[\'sample_f1\'],selected_macro_f1=winner[\'macro_f1\'],bootstrap_95_interval=ci,limitation=\'Single, frozen 3-candidate selection on 78 validation trials; the selected validation score has selection optimism. This is not a Kaggle score. No validation refit or error-specific tuning.\',user_kaggle_baseline=.89423,user_kaggle_leader=.92735)\ne.save_json(result_path,result)\n# Save every frozen submission for reproducibility; main filename always points to selected candidate.\nfor candidate in frozen[\'shortlist\']:\n pp=np.load(\'predictions/locked-\'+candidate[\'name\']+\'.npz\')[\'p\'];hh=e.decisions(pp,candidate[\'rule\'])\n sub=pd.DataFrame({\'ID\':test.ID,\'Predicted\':[\' \'.join(l for l,x in zip(e.LABELS,row) if x) for row in hh[len(val):len(val)+len(test)]]})\n sub.to_csv(\'predictions/submission_\'+candidate[\'name\']+\'.csv\',index=False)\n if candidate[\'name\']==winner[\'name\']:sub.to_csv(\'predictions/part1_submission.csv\',index=False)\nfor name,frame,sl in [(\'validation\',val,slice(0,len(val))),(\'test\',test,slice(len(val),len(val)+len(test))),(\'unlabeled_train\',unlab,slice(len(val)+len(test),None))]:\n out=pd.DataFrame(p[sl],columns=[\'p_\'+l for l in e.LABELS]);out.insert(0,\'ID\',frame.ID.to_numpy());out.to_csv(\'predictions/\'+name+\'_probabilities.csv\',index=False)\naug=e.DATA.copy();aug.loc[e.UNLAB,e.LABELS]=hard[len(val)+len(test):];aug[\'label_source\']=\'original\';aug.loc[e.UNLAB,\'label_source\']=\'pseudo_label_\'+winner[\'name\'];aug.to_csv(\'predictions/part1_train_augmented.csv\',index=False)\n# Validation outputs for report only; do not feed these back into model selection.\nv=pd.DataFrame(hard[:len(val)],columns=[\'pred_\'+l for l in e.LABELS]);v.insert(0,\'ID\',val.ID.to_numpy());v[\'sample_f1\']=row_f1\nfor j,l in enumerate(e.LABELS):v[\'true_\'+l]=y[:,j]\nv.to_csv(\'predictions/validation_audit.csv\',index=False)\nsample=pd.read_csv(\'data/part1_sample_submission.csv\');submission=pd.read_csv(\'predictions/part1_submission.csv\')\nassert list(submission.columns)==[\'ID\',\'Predicted\'] and len(submission)==312\nassert submission.ID.tolist()==sample.ID.tolist() and submission.ID.is_unique\nassert submission.Predicted.notna().all() and submission.Predicted.map(lambda s:bool(s) and set(s.split())<=set(e.LABELS)).all()\nassert aug[e.LABELS].isin([0,1]).all().all() and np.array_equal(aug.loc[e.LAB,e.LABELS],e.Y)\ne.save_json(\'experiments/final-checks.json\',dict(submission_rows=312,submission_id_order_matches_sample=True,labels_valid=True,augmented_rows=len(aug),preserved_ground_truth_rows=len(e.LAB),generated_label_rows=len(e.UNLAB),submission_sha256=hashlib.sha256(Path(\'predictions/part1_submission.csv\').read_bytes()).hexdigest()))\nprint(\'SELECTED\',winner[\'name\'],\'SAMPLE F1\',winner[\'sample_f1\'],\'95% interval\',ci,flush=True)\n'
(replica / 'finalize.py').write_text(source)


5952

### freeze_shortlist.py

Complete source:

In [6]:
source = "import json, numpy as np, hashlib, datetime\nfrom pathlib import Path\nimport experiment as e\npath=Path('experiments/frozen-shortlist.json')\nif path.exists():raise SystemExit('Shortlist already frozen; refusing overwrite.')\nfp=json.loads(Path('experiments/ssl_features-ranking.json').read_text())[0]['config']\ncombo=json.loads(Path('experiments/combinations-ranking.json').read_text())[0]['config']\ntarget=dict(model='svc',C=10,gamma=.001,scale='standard',features=['drug_targets'],feature_weight=1.)\nplain=next(x['config'] for x in json.loads(Path('experiments/robust-ranking.json').read_text()) if x['config']['model']=='pseudo')\nmodels=[fp,combo,target,plain];cache={}\nfor f in Path('experiments').glob('*.npz'):\n z=np.load(f,allow_pickle=True)\n if 'record' not in z:continue\n r=z['record'].item()\n if 'seed' in r:cache[(r['id'],r['seed'])]=z['p']\nshortlist=[]\nfor name,weights in [('fingerprint_ssl',[1,0,0,0]),('balanced_ensemble',[.5,.25,.25,0]),('embedding_ssl',[0,0,0,1])]:\n ps=[]\n for seed in [20261104,20261105,20261106]:ps.append(sum(w*cache[(e.config_id(m),seed)] for w,m in zip(weights,models) if w))\n p=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,p)\n c=dict(model='ensemble',members=[m for m,w in zip(models,weights) if w],weights=[w for w in weights if w])\n rec=dict(name=name,config=c,rule=rule,repeated_oof_f1=score,mean_split_f1=float(np.mean([e.metrics(e.Y,z,rule)['sample_f1'] for z in ps])),prediction_strategy='Mean of 15 models: 5 grouped folds x seeds 20261104,20261105,20261106. No validation labels used in fits.')\n shortlist.append(rec);np.savez_compressed('experiments/frozen-oof-'+name+'.npz',p=p,y=e.Y);print(rec,flush=True)\nobj=dict(frozen_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),members=models,shortlist=shortlist,validation_selection='Choose the highest sample-F1 among these three frozen models; if tied within 0.005, choose fingerprint_ssl, then embedding_ssl, then balanced_ensemble. No subsequent validation-error-driven tuning.',training_data='Official training file only; validation remains held out even for final submission.',engine_sha256=hashlib.sha256(Path('experiment.py').read_bytes()).hexdigest())\ne.save_json(path,obj)\n"
(replica / 'freeze_shortlist.py').write_text(source)


2189

### fullsize_cv.py

Complete source:

In [7]:
source = '"""Nearly-full-training-size audit using leave-one-embedding-group-out CV. No validation access."""\nimport experiment as e,numpy as np,json\nfrom pathlib import Path\nfrom concurrent.futures import ThreadPoolExecutor\nfrom threadpoolctl import threadpool_limits\nfrozen=json.loads(Path(\'experiments/frozen-shortlist.json\').read_text());members=frozen[\'members\']\n# The selection candidates are unchanged in structure; this audit addresses training-size and decoding.\nunique=np.unique(e.GROUP[e.LAB]);allpos=np.arange(len(e.Y))\ndef run_member(c):\n cid=e.config_id(c);out=Path(\'experiments\')/(\'fullsize-\'+cid+\'.npz\')\n if out.exists():return\n p=np.zeros_like(e.Y,float)\n for gi,g in enumerate(unique):\n  b=np.flatnonzero(e.GROUP[e.LAB]==g);a=allpos[~np.isin(allpos,b)]\n  p[b]=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=[g])\n  if gi%25==0:print(\'LEAVE-GROUP-OUT\',cid,gi,len(unique),flush=True)\n rule,score=e.choose_rule(e.Y,p)\n e.save_json(\'experiments/fullsize-\'+cid+\'.json\',dict(config=c,rule=rule,score=score,n_groups=len(unique),mean_training_size=float(np.mean([len(e.Y)-np.sum(e.GROUP[e.LAB]==g) for g in unique]))))\n np.savez_compressed(out,p=p);print(\'DONE\',cid,score,rule,flush=True)\nwith threadpool_limits(limits=1):\n with ThreadPoolExecutor(max_workers=2) as pool:list(pool.map(run_member,members))\nresults=[]\nfor candidate in frozen[\'shortlist\']:\n c=candidate[\'config\'];w=np.array(c[\'weights\'],float);w/=w.sum()\n p=sum(wi*np.load(\'experiments/fullsize-\'+e.config_id(m)+\'.npz\')[\'p\'] for wi,m in zip(w,c[\'members\']))\n rule,score=e.choose_rule(e.Y,p);r=dict(name=candidate[\'name\'],config=c,rule=rule,fullsize_oof_f1=score,metrics=e.metrics(e.Y,p,rule));results.append(r)\n np.savez_compressed(\'experiments/fullsize-oof-\'+candidate[\'name\']+\'.npz\',p=p,y=e.Y)\ne.save_json(\'experiments/fullsize-ranking.json\',sorted(results,key=lambda r:r[\'fullsize_oof_f1\'],reverse=True))\nprint(\'RESULTS\',results,flush=True)\n'
(replica / 'fullsize_cv.py').write_text(source)


1913

### make_candidate.py

Complete source:

In [8]:
source = '"""Regenerate the final train-only full-fit candidate. Never reads validation labels."""\nfrom pathlib import Path\nimport json,hashlib,datetime\nimport numpy as np,pandas as pd\nimport experiment as e\nfrom threadpoolctl import threadpool_limits\nrank=json.loads(Path(\'experiments/fullsize-ranking.json\').read_text())\nselected=rank[0]\ntest=pd.read_csv(\'data/part1_test.csv\');u=e.DATA.iloc[e.UNLAB].drop(columns=e.LABELS)\nquery=pd.concat([test,u],ignore_index=True)\nwith threadpool_limits(limits=1):p=e.predict(np.arange(len(e.Y)),query,selected[\'config\'])\nh=e.decisions(p,selected[\'rule\'])\nsub=pd.DataFrame({\'ID\':test.ID,\'Predicted\':[\' \'.join(l for l,v in zip(e.LABELS,row) if v) for row in h[:len(test)]]})\npath=Path(\'predictions/part1_fullfit_candidate.csv\');sub.to_csv(path,index=False)\nprobs=pd.DataFrame(p[:len(test)],columns=[\'p_\'+l for l in e.LABELS]);probs.insert(0,\'ID\',test.ID);probs.to_csv(\'predictions/fullfit_test_probabilities.csv\',index=False)\naug=e.DATA.copy();aug.loc[e.UNLAB,e.LABELS]=h[len(test):];aug[\'label_source\']=\'original\';aug.loc[e.UNLAB,\'label_source\']=\'fullfit_pseudo_label\';aug.to_csv(\'predictions/part1_train_augmented_fullfit.csv\',index=False)\nup=pd.DataFrame(p[len(test):],columns=[\'p_\'+l for l in e.LABELS]);up.insert(0,\'ID\',u.ID.to_numpy());up.to_csv(\'predictions/fullfit_unlabeled_probabilities.csv\',index=False)\nassert sub.ID.tolist()==pd.read_csv(\'data/part1_sample_submission.csv\').ID.tolist()\nassert sub.Predicted.map(lambda s:bool(s) and set(s.split())<=set(e.LABELS)).all()\nassert aug[e.LABELS].isin([0,1]).all().all() and np.array_equal(aug.loc[e.LAB,e.LABELS],e.Y)\nmanifest=dict(created_at=datetime.datetime.now(datetime.timezone.utc).isoformat(),selected=selected,training_gold_rows=len(e.Y),unlabeled_rows_used=len(e.UNLAB),test_rows=len(test),prediction_strategy=\'Fit each member on all 157 labeled rows; unsupervised transformations and teacher clustering use the permitted training unlabeled pool. Average member probabilities with weights 0.50 / 0.25 / 0.25; threshold 0.35, fallback to highest probability if empty.\',validation_status=\'Not evaluated again on the used external validation set. Earlier 15-fold-bagged models achieved best 0.850427; their scores do not measure this full-fit candidate.\',kaggle_score=None,submission=str(path),submission_sha256=hashlib.sha256(path.read_bytes()).hexdigest(),engine_sha256=hashlib.sha256(Path(\'experiment.py\').read_bytes()).hexdigest(),claims=\'0.936306 is training-only leave-one-embedding-group-out development F1, not Kaggle F1. The winning-score objective remains unverified.\')\ne.save_json(\'experiments/final-candidate.json\',manifest)\nprint(json.dumps(manifest,indent=2))\n'
(replica / 'make_candidate.py').write_text(source)


2665

### prepare_consensus.py

Complete source:

In [9]:
source = "import experiment as e\nconfigs=[]\nteachers=[dict(model='graph',pca=25,k=5,alpha=.8),dict(model='cluster',cluster='hdbscan',min_cluster=5,pca=25)]\nfor t in teachers:\n for model in ['logistic','svc']:\n  for c in ([.1,.3] if model=='logistic' else [3,10]):\n   for w in [.05,.2]:\n    for conf in [.8,.95]:\n     configs.append(dict(model='pseudo',teacher=t,student=model,C=c,gamma=.001,scale='global',pseudo_weight=w,confidence=conf,labelwise=True,consensus=True))\ne.save_json('experiments/consensus-input.json',configs)\n"
(replica / 'prepare_consensus.py').write_text(source)


516

### prepare_metric.py

Complete source:

In [10]:
source = "import experiment as e\nconfigs=[]\nfor model in ['logistic','svc']:\n for k in [5,15,40]:\n  for residual in [0,.5,2]:\n   configs.append(dict(model='metric_graph',metric_model=model,C=.3 if model=='logistic' else 10,gamma=.001,scale='global',k=k,alpha=.8,residual=residual))\nfor n in [2,3,5,10,20,40]:configs.append(dict(model='pls',components=n))\ne.save_json('experiments/metric-input.json',configs)\n"
(replica / 'prepare_metric.py').write_text(source)


398

### prepare_refinement.py

Complete source:

In [11]:
source = "import experiment as e,json\nfrom pathlib import Path\nrows=[json.loads(x) for x in Path('experiments/trials.jsonl').read_text().splitlines()];ok=[r for r in rows if r['status']=='ok'];configs=[]\nfor scale in ['standard','global']:\n for C in [.03,.05,.2,.3,.5,2,3,30]:configs.append(dict(model='logistic',scale=scale,C=C))\n for C in [1,3,10,30]:\n  for gamma in [.0001,.0003,.0005,.002,.005]:configs.append(dict(model='svc',scale=scale,C=C,gamma=gamma))\nfor C in [1,3,30,100]:\n for scale in ['global','raw']:configs.append(dict(model='logistic',scale=scale,norm=True,C=C))\nfor alpha in [.001,.01,.1,1]:\n for gamma in [.0003,.001,.003,.01]:configs.append(dict(model='krr',alpha=alpha,gamma=gamma))\nfor amplitude in [1,10]:\n for length in [5,10,20,40]:configs.append(dict(model='gp',amplitude=amplitude,length=length))\nfor pc in [2,3,5]:configs.append(dict(model='qda',pca=pc,reg=.9))\nfor pc in [5,15,30]:\n for k in [3,7]:configs.append(dict(model='nca',pca=30,components=pc,k=k))\n# Refine graph geometry, especially sharper local neighborhoods and less noisy dimensions.\nfor pc in [5,10,15,40]:\n for k in [3,7,12]:configs.append(dict(model='graph',pca=pc,k=k,alpha=.8))\nfor scale in ['global','raw']:\n for norm in [True,False]:\n  for k in [3,7,15]:configs.append(dict(model='graph',scale=scale,norm=norm,k=k,alpha=.9))\nfor typ in ['kmeans','gmm']:\n for pc in [5,10,50]:\n  for n in [5,15,35]:configs.append(dict(model='cluster',cluster=typ,pca=pc,clusters=n))\n# Pseudo-labels generated by graph or density-cluster teachers; small weight prevents false labels dominating.\nteachers=[max([r for r in ok if r['model']==t],key=lambda r:r['score'])['config'] for t in ['graph','cluster']]\nteachers.append(dict(model='cluster',cluster='kmeans',clusters=12,pca=25))\nfor teacher in teachers:\n for student in ['svc','logistic']:\n  for weight in [.05,.2,.5]:\n   for confidence in [.8,.95]:configs.append(dict(model='pseudo',teacher=teacher,student=student,C=10 if student=='svc' else .1,gamma=.001,pseudo_weight=weight,confidence=confidence,labelwise=True))\ne.save_json('experiments/refine-input.json',configs)\n# Stage independently adds each semantic group to the embedding baseline.\nblocks=['year','criteria','criteria_char','diseases_cleaned','drugs','icdcodes','matched_drugs','drug_targets','drug_actions','drug_pathways','pathway_level0','pathway_level2','pathway_levels','fingerprint','descriptors','missingness','counts']\nconfigs=[]\nfor base in [dict(model='svc',C=10,gamma=.001,scale='standard'),dict(model='logistic',C=.1,scale='global')]:\n for block in blocks:\n  for w in [.15,.5,1.]:configs.append({**base,'features':[block],'feature_weight':w})\ne.save_json('experiments/features-input.json',configs)\n"
(replica / 'prepare_refinement.py').write_text(source)


2697

### robustness.py

Complete source:

In [12]:
source = '"""Fresh-seed checks and predeclared feature combinations; no external validation."""\nimport json\nfrom pathlib import Path\nimport experiment as e\nrs=[json.loads(s) for s in Path(\'experiments/trials.jsonl\').read_text().splitlines()]\nok=[r for r in rs if r[\'status\']==\'ok\' and r[\'tag\'] in [\'screen\',\'refine\',\'features\',\'consensus\']]\nchosen=[]\nfor model in [\'svc\',\'logistic\',\'lda\',\'chain\',\'powerset\',\'graph\',\'cluster\',\'pseudo\']:\n matches=[r for r in ok if r[\'model\']==model and not r[\'config\'].get(\'features\')]\n chosen.extend(sorted(matches,key=lambda x:x[\'score\'],reverse=True)[:(3 if model in [\'svc\',\'logistic\',\'pseudo\'] else 1)])\nfor block in [\'drugs\',\'pathway_levels\',\'pathway_level0\',\'descriptors\',\'fingerprint\',\'criteria\',\'drug_pathways\',\'drug_targets\',\'year\']:\n matches=[r for r in ok if r[\'config\'].get(\'features\')==[block]]\n chosen.append(max(matches,key=lambda x:x[\'score\']))\nconfigs=list({e.config_id(r[\'config\']):r[\'config\'] for r in chosen}.values())\ne.save_json(\'experiments/robustness-input.json\',configs)\nfor seed in [20261105,20261106,20261107]:\n for i,c in enumerate(configs):\n  print(\'REPEAT\',seed,i+1,len(configs),c,flush=True);e.evaluate(c,seed=seed,tag=\'robustness\')\nbase=dict(model=\'svc\',C=10,gamma=.001,scale=\'standard\')\ncombos=[[\'drugs\',\'pathway_levels\'],[\'drugs\',\'descriptors\'],[\'drugs\',\'pathway_level0\'],[\'drugs\',\'criteria\'],[\'pathway_levels\',\'descriptors\'],[\'drugs\',\'pathway_levels\',\'descriptors\'],[\'drugs\',\'pathway_levels\',\'criteria\'],[\'drugs\',\'pathway_levels\',\'descriptors\',\'criteria\'],[\'criteria\',\'diseases_cleaned\',\'drugs\',\'icdcodes\',\'matched_drugs\',\'drug_targets\',\'drug_actions\',\'drug_pathways\',\'fingerprint\',\'descriptors\',\'year\',\'missingness\',\'counts\']]\nconfigs=[{**base,\'features\':b,\'feature_weight\':w} for b in combos for w in [.15,.5]]\ne.save_json(\'experiments/combinations-input.json\',configs)\nfor seed in [20261104,20261105,20261106]:\n for c in configs:e.evaluate(c,seed=seed,tag=\'combinations\')\n'
(replica / 'robustness.py').write_text(source)


1932

### select_ensembles.py

Complete source:

In [13]:
source = "import experiment as e,json,numpy as np\nfrom pathlib import Path\nfrom collections import defaultdict\nby=defaultdict(dict)\nfor f in Path('experiments').glob('*.npz'):\n z=np.load(f,allow_pickle=True);r=z['record'].item()\n if r['tag'] in ['screen','refine','features','consensus','robustness','combinations']:by[r['id']][r['seed']]=(r,z['p'])\nrank=json.loads(Path('experiments/robust-ranking.json').read_text())\n# Broad mechanisms rather than many near-identical hyperparameter fits.\nproto=[\n dict(model='logistic',scale='global',C=.2),\n dict(model='logistic',scale='global',C=.3),\n dict(model='svc',C=10,gamma=.001,scale='standard',features=['drug_targets'],feature_weight=1.),\n dict(model='svc',C=10,gamma=.001,scale='standard',features=['fingerprint'],feature_weight=.5),\n dict(model='svc',C=10,gamma=.001,scale='standard',features=['criteria'],feature_weight=1.),\n next(x['config'] for x in rank if x['config']['model']=='pseudo'),\n next(x['config'] for x in rank if x['config']['model']=='pseudo' and x['config']['teacher']['model']=='graph'),\n]\nensembles=[]\nfor j in [2,3,4,5,6]:\n for w in [.25,.5,.75]:ensembles.append(dict(model='ensemble',members=[proto[0],proto[j]],weights=[1-w,w]))\nfor ids in [[0,2,5],[0,3,5],[0,2,4,5],[0,2,3,4,5],[0,5,6],[2,4,5],[0,1,2,3,4,5,6]]:\n ensembles.append(dict(model='ensemble',members=[proto[j] for j in ids]))\nresults=[]\nfor c in ensembles:\n ids=[e.config_id(m) for m in c['members']];seeds=sorted(set.intersection(*(set(by[cid]) for cid in ids)))\n ps=[];weights=np.asarray(c.get('weights',np.ones(len(ids))),float);weights/=weights.sum()\n for seed in seeds:ps.append(sum(w*by[cid][seed][1] for cid,w in zip(ids,weights)))\n p=np.mean(ps,axis=0);rule,score=e.choose_rule(e.Y,p)\n r=dict(config=c,id=e.config_id(c),rule=rule,averaged_oof_score=score,mean_split_score=float(np.mean([e.metrics(e.Y,x,rule)['sample_f1'] for x in ps])),n_seeds=len(seeds))\n results.append(r)\n np.savez_compressed('experiments/ensemble-'+r['id']+'.npz',p=p,record=r)\nresults.sort(key=lambda r:(r['averaged_oof_score'],r['mean_split_score']),reverse=True)\ne.save_json('experiments/ensemble-ranking.json',results)\nfor r in results[:8]:print(r['averaged_oof_score'],r['mean_split_score'],r['rule'],r['config'])\n# Direct SSL candidate with feature additions, evaluated across fresh seeds.\nbase=proto[5];configs=[]\nfor block in ['drug_targets','drugs','criteria','fingerprint','pathway_levels']:\n for w in [.15,.5]:configs.append({**base,'features':[block],'feature_weight':w})\ne.save_json('experiments/ssl-features-input.json',configs)\n"
(replica / 'select_ensembles.py').write_text(source)


2546

### verify_engine.py

Complete source:

In [14]:
source = '"""Mathematical and leakage checks for custom experiment plumbing."""\nimport numpy as np\nimport experiment as e\nfrom threadpoolctl import threadpool_limits\nfrom sklearn.metrics import f1_score\nrng=np.random.default_rng(42)\nwith threadpool_limits(limits=1):\n    for a,b in e.make_splits():\n        assert len(set(e.GROUP[e.LAB[a]])&set(e.GROUP[e.LAB[b]]))==0\n    # Deterministic posteriors must produce precisely their nonempty labels.\n    patterns=((np.arange(1,32)[:,None]>>np.arange(5))&1)\n    assert np.array_equal(e.decisions(patterns,\'expected_f1\'),patterns)\n    # Graph must respect two well-separated components, and probabilities sum appropriately.\n    X=np.r_[rng.normal(-10,.1,(20,3)),rng.normal(10,.1,(20,3))]\n    Y=np.full((40,5),-1);Y[0]=[1,0,0,0,0];Y[20]=[0,1,0,0,0]\n    for hard in [False,True]:\n        m=e.Graph({\'k\':5,\'alpha\':.95,\'hard\':hard}).fit(X,Y)\n        p=m.predict(np.array([[-10]*3,[10]*3]))\n        # Hard mode deliberately shrinks toward the global prior, unlike normalized spreading.\n        assert p[0,0]>(.6 if hard else .95) and p[1,1]>(.6 if hard else .95)\n        assert np.all((p>=0)&(p<=1))\n    a,b=e.make_splits()[0]\n    c={\'model\':\'logistic\',\'C\':.1}\n    p=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])\n    original=e.Y[b].copy();e.Y[b]=1-original\n    q=e.predict(a,e.DATA.iloc[e.LAB[b]],c,excluded_groups=e.GROUP[e.LAB[b]])\n    e.Y[b]=original\n    assert np.allclose(p,q), \'Held-out labels affected fitting\'\n    query=e.DATA.iloc[e.LAB[b]].copy();query[\'ID\']=\'arbitrary\'\n    assert np.allclose(p,e.predict(a,query,c,excluded_groups=e.GROUP[e.LAB[b]])), \'ID affected predictions\'\n    # Scaler fit is independent of query batch; changing one query cannot change the others.\n    altered=query.copy();altered.loc[altered.index[0],e.EMB]=100000\n    r=e.predict(a,altered,c,excluded_groups=e.GROUP[e.LAB[b]])\n    assert np.allclose(p[1:],r[1:]), \'Query contaminated fit\'\nprint(\'PASS: grouped folds, expected-F1 decoder, graph diffusion, held-out-label isolation, ID exclusion, preprocessing isolation\')\n'
(replica / 'verify_engine.py').write_text(source)


2061

In [15]:
static_configs = {'consensus-input.json': [{'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True, 'consensus': True}], 'combinations-input.json': [{'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'descriptors'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'descriptors'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_level0'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_level0'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'criteria'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'criteria'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_levels', 'descriptors'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_levels', 'descriptors'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels', 'descriptors'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels', 'descriptors'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels', 'criteria'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels', 'criteria'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels', 'descriptors', 'criteria'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs', 'pathway_levels', 'descriptors', 'criteria'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria', 'diseases_cleaned', 'drugs', 'icdcodes', 'matched_drugs', 'drug_targets', 'drug_actions', 'drug_pathways', 'fingerprint', 'descriptors', 'year', 'missingness', 'counts'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria', 'diseases_cleaned', 'drugs', 'icdcodes', 'matched_drugs', 'drug_targets', 'drug_actions', 'drug_pathways', 'fingerprint', 'descriptors', 'year', 'missingness', 'counts'], 'feature_weight': 0.5}], 'dictionary-input.json': [{'model': 'cluster_features', 'clusters': 10, 'C': 0.01, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 0.01, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 0.1, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 0.1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 1, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 10, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 10, 'C': 10, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 0.01, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 0.01, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 0.1, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 0.1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 1, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 10, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 10, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 0.01, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 0.01, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 0.1, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 0.1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 1, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 10, 'band': 0.5, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 80, 'C': 10, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 0.1, 'band': 0.5, 'original_weight': 1.0, 'pca': 25}, {'model': 'cluster_features', 'clusters': 30, 'C': 1, 'band': 0.5, 'original_weight': 1.0, 'pca': 25}, {'model': 'cluster_features', 'clusters': 80, 'C': 0.1, 'band': 0.5, 'original_weight': 1.0, 'pca': 25}, {'model': 'cluster_features', 'clusters': 80, 'C': 1, 'band': 0.5, 'original_weight': 1.0, 'pca': 25}], 'phase2-repeat-input.json': [{'model': 'softmax', 'C': 0.3, 'scale': 'global'}, {'model': 'softmax', 'C': 1, 'scale': 'standard'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.001, 'scale': 'global'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.003, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 1, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.1, 'scale': 'global'}, {'model': 'semigmm', 'pca': 25, 'label_weight': 20, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 5, 'cov': 'class', 'shrink': 0.8}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 0.1, 'mu': 0.01}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 0.1, 'mu': 0.01}, {'model': 'cluster_features', 'clusters': 10, 'C': 1, 'band': 1.0, 'original_weight': 0.3}, {'model': 'cluster_features', 'clusters': 30, 'C': 10, 'band': 1.0, 'original_weight': 0.3}], 'ssl-features-input.json': [{'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['drug_targets'], 'feature_weight': 0.15}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['drug_targets'], 'feature_weight': 0.5}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['drugs'], 'feature_weight': 0.15}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['drugs'], 'feature_weight': 0.5}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['criteria'], 'feature_weight': 0.15}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['criteria'], 'feature_weight': 0.5}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['fingerprint'], 'feature_weight': 0.15}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['fingerprint'], 'feature_weight': 0.5}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['pathway_levels'], 'feature_weight': 0.15}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True, 'features': ['pathway_levels'], 'feature_weight': 0.5}], 'robustness-input.json': [{'model': 'svc', 'scale': 'global', 'C': 10, 'gamma': 0.0001}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard'}, {'model': 'svc', 'scale': 'global', 'C': 30, 'gamma': 0.0003}, {'model': 'logistic', 'scale': 'global', 'C': 0.3}, {'model': 'logistic', 'scale': 'global', 'C': 0.2}, {'model': 'logistic', 'scale': 'global', 'norm': True, 'C': 100}, {'model': 'lda'}, {'model': 'chain'}, {'model': 'powerset'}, {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True, 'consensus': True}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_levels'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level0'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['descriptors'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['fingerprint'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_pathways'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_targets'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['year'], 'feature_weight': 0.15}], 'metric-input.json': [{'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 5, 'alpha': 0.8, 'residual': 0}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 5, 'alpha': 0.8, 'residual': 0.5}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 5, 'alpha': 0.8, 'residual': 2}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 15, 'alpha': 0.8, 'residual': 0}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 15, 'alpha': 0.8, 'residual': 0.5}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 15, 'alpha': 0.8, 'residual': 2}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 40, 'alpha': 0.8, 'residual': 0}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 40, 'alpha': 0.8, 'residual': 0.5}, {'model': 'metric_graph', 'metric_model': 'logistic', 'C': 0.3, 'gamma': 0.001, 'scale': 'global', 'k': 40, 'alpha': 0.8, 'residual': 2}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 5, 'alpha': 0.8, 'residual': 0}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 5, 'alpha': 0.8, 'residual': 0.5}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 5, 'alpha': 0.8, 'residual': 2}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 15, 'alpha': 0.8, 'residual': 0}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 15, 'alpha': 0.8, 'residual': 0.5}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 15, 'alpha': 0.8, 'residual': 2}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 40, 'alpha': 0.8, 'residual': 0}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 40, 'alpha': 0.8, 'residual': 0.5}, {'model': 'metric_graph', 'metric_model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'global', 'k': 40, 'alpha': 0.8, 'residual': 2}, {'model': 'pls', 'components': 2}, {'model': 'pls', 'components': 3}, {'model': 'pls', 'components': 5}, {'model': 'pls', 'components': 10}, {'model': 'pls', 'components': 20}, {'model': 'pls', 'components': 40}], 'phase2-input.json': [{'model': 'softmax', 'C': 0.003, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.003, 'scale': 'standard'}, {'model': 'softmax', 'C': 0.01, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.01, 'scale': 'standard'}, {'model': 'softmax', 'C': 0.03, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.03, 'scale': 'standard'}, {'model': 'softmax', 'C': 0.1, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.1, 'scale': 'standard'}, {'model': 'softmax', 'C': 0.3, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.3, 'scale': 'standard'}, {'model': 'softmax', 'C': 1, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 1, 'scale': 'standard'}, {'model': 'softmax', 'C': 3, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 3, 'scale': 'standard'}, {'model': 'softmax', 'C': 10, 'scale': 'standard'}, {'model': 'powerset', 'kernel': 'linear', 'C': 10, 'scale': 'standard'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.0003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.001, 'scale': 'standard'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.01, 'scale': 'standard'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.0003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.001, 'scale': 'standard'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.01, 'scale': 'standard'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.0003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.001, 'scale': 'standard'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.01, 'scale': 'standard'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.0003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.001, 'scale': 'standard'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.01, 'scale': 'standard'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.0003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.001, 'scale': 'standard'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.003, 'scale': 'standard'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.01, 'scale': 'standard'}, {'model': 'softmax', 'C': 0.003, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.003, 'scale': 'global'}, {'model': 'softmax', 'C': 0.01, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.01, 'scale': 'global'}, {'model': 'softmax', 'C': 0.03, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.03, 'scale': 'global'}, {'model': 'softmax', 'C': 0.1, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.1, 'scale': 'global'}, {'model': 'softmax', 'C': 0.3, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 0.3, 'scale': 'global'}, {'model': 'softmax', 'C': 1, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 1, 'scale': 'global'}, {'model': 'softmax', 'C': 3, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 3, 'scale': 'global'}, {'model': 'softmax', 'C': 10, 'scale': 'global'}, {'model': 'powerset', 'kernel': 'linear', 'C': 10, 'scale': 'global'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.0003, 'scale': 'global'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.001, 'scale': 'global'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.003, 'scale': 'global'}, {'model': 'softsvc', 'C': 0.3, 'gamma': 0.01, 'scale': 'global'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.0003, 'scale': 'global'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.001, 'scale': 'global'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.003, 'scale': 'global'}, {'model': 'softsvc', 'C': 1, 'gamma': 0.01, 'scale': 'global'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.0003, 'scale': 'global'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.001, 'scale': 'global'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.003, 'scale': 'global'}, {'model': 'softsvc', 'C': 3, 'gamma': 0.01, 'scale': 'global'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.0003, 'scale': 'global'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.001, 'scale': 'global'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.003, 'scale': 'global'}, {'model': 'softsvc', 'C': 10, 'gamma': 0.01, 'scale': 'global'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.0003, 'scale': 'global'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.001, 'scale': 'global'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.003, 'scale': 'global'}, {'model': 'softsvc', 'C': 30, 'gamma': 0.01, 'scale': 'global'}, {'model': 'semigmm', 'pca': 5, 'label_weight': 1, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 5, 'label_weight': 1, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 5, 'label_weight': 5, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 5, 'label_weight': 5, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 5, 'label_weight': 20, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 5, 'label_weight': 20, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 10, 'label_weight': 1, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 10, 'label_weight': 1, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 10, 'label_weight': 5, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 10, 'label_weight': 5, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 10, 'label_weight': 20, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 10, 'label_weight': 20, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 1, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 1, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 5, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 5, 'cov': 'class', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 20, 'cov': 'shared', 'shrink': 0.8}, {'model': 'semigmm', 'pca': 25, 'label_weight': 20, 'cov': 'class', 'shrink': 0.8}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 0.1, 'mu': 0.01}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 0.1, 'mu': 0.1}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 0.1, 'mu': 1}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 1, 'mu': 0.01}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 1, 'mu': 0.1}, {'model': 'laprls', 'gamma': 0.001, 'alpha': 1, 'mu': 1}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 0.1, 'mu': 0.01}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 0.1, 'mu': 0.1}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 0.1, 'mu': 1}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 1, 'mu': 0.01}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 1, 'mu': 0.1}, {'model': 'laprls', 'gamma': 0.003, 'alpha': 1, 'mu': 1}], 'refine-input.json': [{'model': 'logistic', 'scale': 'standard', 'C': 0.03}, {'model': 'logistic', 'scale': 'standard', 'C': 0.05}, {'model': 'logistic', 'scale': 'standard', 'C': 0.2}, {'model': 'logistic', 'scale': 'standard', 'C': 0.3}, {'model': 'logistic', 'scale': 'standard', 'C': 0.5}, {'model': 'logistic', 'scale': 'standard', 'C': 2}, {'model': 'logistic', 'scale': 'standard', 'C': 3}, {'model': 'logistic', 'scale': 'standard', 'C': 30}, {'model': 'svc', 'scale': 'standard', 'C': 1, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'standard', 'C': 1, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'standard', 'C': 1, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'standard', 'C': 1, 'gamma': 0.002}, {'model': 'svc', 'scale': 'standard', 'C': 1, 'gamma': 0.005}, {'model': 'svc', 'scale': 'standard', 'C': 3, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'standard', 'C': 3, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'standard', 'C': 3, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'standard', 'C': 3, 'gamma': 0.002}, {'model': 'svc', 'scale': 'standard', 'C': 3, 'gamma': 0.005}, {'model': 'svc', 'scale': 'standard', 'C': 10, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'standard', 'C': 10, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'standard', 'C': 10, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'standard', 'C': 10, 'gamma': 0.002}, {'model': 'svc', 'scale': 'standard', 'C': 10, 'gamma': 0.005}, {'model': 'svc', 'scale': 'standard', 'C': 30, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'standard', 'C': 30, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'standard', 'C': 30, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'standard', 'C': 30, 'gamma': 0.002}, {'model': 'svc', 'scale': 'standard', 'C': 30, 'gamma': 0.005}, {'model': 'logistic', 'scale': 'global', 'C': 0.03}, {'model': 'logistic', 'scale': 'global', 'C': 0.05}, {'model': 'logistic', 'scale': 'global', 'C': 0.2}, {'model': 'logistic', 'scale': 'global', 'C': 0.3}, {'model': 'logistic', 'scale': 'global', 'C': 0.5}, {'model': 'logistic', 'scale': 'global', 'C': 2}, {'model': 'logistic', 'scale': 'global', 'C': 3}, {'model': 'logistic', 'scale': 'global', 'C': 30}, {'model': 'svc', 'scale': 'global', 'C': 1, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'global', 'C': 1, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'global', 'C': 1, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'global', 'C': 1, 'gamma': 0.002}, {'model': 'svc', 'scale': 'global', 'C': 1, 'gamma': 0.005}, {'model': 'svc', 'scale': 'global', 'C': 3, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'global', 'C': 3, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'global', 'C': 3, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'global', 'C': 3, 'gamma': 0.002}, {'model': 'svc', 'scale': 'global', 'C': 3, 'gamma': 0.005}, {'model': 'svc', 'scale': 'global', 'C': 10, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'global', 'C': 10, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'global', 'C': 10, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'global', 'C': 10, 'gamma': 0.002}, {'model': 'svc', 'scale': 'global', 'C': 10, 'gamma': 0.005}, {'model': 'svc', 'scale': 'global', 'C': 30, 'gamma': 0.0001}, {'model': 'svc', 'scale': 'global', 'C': 30, 'gamma': 0.0003}, {'model': 'svc', 'scale': 'global', 'C': 30, 'gamma': 0.0005}, {'model': 'svc', 'scale': 'global', 'C': 30, 'gamma': 0.002}, {'model': 'svc', 'scale': 'global', 'C': 30, 'gamma': 0.005}, {'model': 'logistic', 'scale': 'global', 'norm': True, 'C': 1}, {'model': 'logistic', 'scale': 'raw', 'norm': True, 'C': 1}, {'model': 'logistic', 'scale': 'global', 'norm': True, 'C': 3}, {'model': 'logistic', 'scale': 'raw', 'norm': True, 'C': 3}, {'model': 'logistic', 'scale': 'global', 'norm': True, 'C': 30}, {'model': 'logistic', 'scale': 'raw', 'norm': True, 'C': 30}, {'model': 'logistic', 'scale': 'global', 'norm': True, 'C': 100}, {'model': 'logistic', 'scale': 'raw', 'norm': True, 'C': 100}, {'model': 'krr', 'alpha': 0.001, 'gamma': 0.0003}, {'model': 'krr', 'alpha': 0.001, 'gamma': 0.001}, {'model': 'krr', 'alpha': 0.001, 'gamma': 0.003}, {'model': 'krr', 'alpha': 0.001, 'gamma': 0.01}, {'model': 'krr', 'alpha': 0.01, 'gamma': 0.0003}, {'model': 'krr', 'alpha': 0.01, 'gamma': 0.001}, {'model': 'krr', 'alpha': 0.01, 'gamma': 0.003}, {'model': 'krr', 'alpha': 0.01, 'gamma': 0.01}, {'model': 'krr', 'alpha': 0.1, 'gamma': 0.0003}, {'model': 'krr', 'alpha': 0.1, 'gamma': 0.001}, {'model': 'krr', 'alpha': 0.1, 'gamma': 0.003}, {'model': 'krr', 'alpha': 0.1, 'gamma': 0.01}, {'model': 'krr', 'alpha': 1, 'gamma': 0.0003}, {'model': 'krr', 'alpha': 1, 'gamma': 0.001}, {'model': 'krr', 'alpha': 1, 'gamma': 0.003}, {'model': 'krr', 'alpha': 1, 'gamma': 0.01}, {'model': 'gp', 'amplitude': 1, 'length': 5}, {'model': 'gp', 'amplitude': 1, 'length': 10}, {'model': 'gp', 'amplitude': 1, 'length': 20}, {'model': 'gp', 'amplitude': 1, 'length': 40}, {'model': 'gp', 'amplitude': 10, 'length': 5}, {'model': 'gp', 'amplitude': 10, 'length': 10}, {'model': 'gp', 'amplitude': 10, 'length': 20}, {'model': 'gp', 'amplitude': 10, 'length': 40}, {'model': 'qda', 'pca': 2, 'reg': 0.9}, {'model': 'qda', 'pca': 3, 'reg': 0.9}, {'model': 'qda', 'pca': 5, 'reg': 0.9}, {'model': 'nca', 'pca': 30, 'components': 5, 'k': 3}, {'model': 'nca', 'pca': 30, 'components': 5, 'k': 7}, {'model': 'nca', 'pca': 30, 'components': 15, 'k': 3}, {'model': 'nca', 'pca': 30, 'components': 15, 'k': 7}, {'model': 'nca', 'pca': 30, 'components': 30, 'k': 3}, {'model': 'nca', 'pca': 30, 'components': 30, 'k': 7}, {'model': 'graph', 'pca': 5, 'k': 3, 'alpha': 0.8}, {'model': 'graph', 'pca': 5, 'k': 7, 'alpha': 0.8}, {'model': 'graph', 'pca': 5, 'k': 12, 'alpha': 0.8}, {'model': 'graph', 'pca': 10, 'k': 3, 'alpha': 0.8}, {'model': 'graph', 'pca': 10, 'k': 7, 'alpha': 0.8}, {'model': 'graph', 'pca': 10, 'k': 12, 'alpha': 0.8}, {'model': 'graph', 'pca': 15, 'k': 3, 'alpha': 0.8}, {'model': 'graph', 'pca': 15, 'k': 7, 'alpha': 0.8}, {'model': 'graph', 'pca': 15, 'k': 12, 'alpha': 0.8}, {'model': 'graph', 'pca': 40, 'k': 3, 'alpha': 0.8}, {'model': 'graph', 'pca': 40, 'k': 7, 'alpha': 0.8}, {'model': 'graph', 'pca': 40, 'k': 12, 'alpha': 0.8}, {'model': 'graph', 'scale': 'global', 'norm': True, 'k': 3, 'alpha': 0.9}, {'model': 'graph', 'scale': 'global', 'norm': True, 'k': 7, 'alpha': 0.9}, {'model': 'graph', 'scale': 'global', 'norm': True, 'k': 15, 'alpha': 0.9}, {'model': 'graph', 'scale': 'global', 'norm': False, 'k': 3, 'alpha': 0.9}, {'model': 'graph', 'scale': 'global', 'norm': False, 'k': 7, 'alpha': 0.9}, {'model': 'graph', 'scale': 'global', 'norm': False, 'k': 15, 'alpha': 0.9}, {'model': 'graph', 'scale': 'raw', 'norm': True, 'k': 3, 'alpha': 0.9}, {'model': 'graph', 'scale': 'raw', 'norm': True, 'k': 7, 'alpha': 0.9}, {'model': 'graph', 'scale': 'raw', 'norm': True, 'k': 15, 'alpha': 0.9}, {'model': 'graph', 'scale': 'raw', 'norm': False, 'k': 3, 'alpha': 0.9}, {'model': 'graph', 'scale': 'raw', 'norm': False, 'k': 7, 'alpha': 0.9}, {'model': 'graph', 'scale': 'raw', 'norm': False, 'k': 15, 'alpha': 0.9}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 5, 'clusters': 5}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 5, 'clusters': 15}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 5, 'clusters': 35}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 10, 'clusters': 5}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 10, 'clusters': 15}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 10, 'clusters': 35}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 50, 'clusters': 5}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 50, 'clusters': 15}, {'model': 'cluster', 'cluster': 'kmeans', 'pca': 50, 'clusters': 35}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 5, 'clusters': 5}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 5, 'clusters': 15}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 5, 'clusters': 35}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 10, 'clusters': 5}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 10, 'clusters': 15}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 10, 'clusters': 35}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 50, 'clusters': 5}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 50, 'clusters': 15}, {'model': 'cluster', 'cluster': 'gmm', 'pca': 50, 'clusters': 35}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'graph', 'pca': 25, 'k': 5, 'alpha': 0.8}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'hdbscan', 'min_cluster': 5, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'svc', 'C': 10, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.05, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.2, 'confidence': 0.95, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.8, 'labelwise': True}, {'model': 'pseudo', 'teacher': {'model': 'cluster', 'cluster': 'kmeans', 'clusters': 12, 'pca': 25}, 'student': 'logistic', 'C': 0.1, 'gamma': 0.001, 'pseudo_weight': 0.5, 'confidence': 0.95, 'labelwise': True}], 'features-input.json': [{'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['year'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['year'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['year'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria_char'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria_char'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['criteria_char'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['diseases_cleaned'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['diseases_cleaned'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['diseases_cleaned'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drugs'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['icdcodes'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['icdcodes'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['icdcodes'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['matched_drugs'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['matched_drugs'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['matched_drugs'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_targets'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_targets'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_targets'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_actions'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_actions'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_actions'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_pathways'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_pathways'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['drug_pathways'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level0'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level0'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level0'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level2'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level2'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_level2'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_levels'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_levels'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['pathway_levels'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['fingerprint'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['fingerprint'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['fingerprint'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['descriptors'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['descriptors'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['descriptors'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['missingness'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['missingness'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['missingness'], 'feature_weight': 1.0}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['counts'], 'feature_weight': 0.15}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['counts'], 'feature_weight': 0.5}, {'model': 'svc', 'C': 10, 'gamma': 0.001, 'scale': 'standard', 'features': ['counts'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['year'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['year'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['year'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['criteria'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['criteria'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['criteria'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['criteria_char'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['criteria_char'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['criteria_char'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['diseases_cleaned'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['diseases_cleaned'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['diseases_cleaned'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drugs'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drugs'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drugs'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['icdcodes'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['icdcodes'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['icdcodes'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['matched_drugs'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['matched_drugs'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['matched_drugs'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_targets'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_targets'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_targets'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_actions'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_actions'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_actions'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_pathways'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_pathways'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['drug_pathways'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_level0'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_level0'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_level0'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_level2'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_level2'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_level2'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_levels'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_levels'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['pathway_levels'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['fingerprint'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['fingerprint'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['fingerprint'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['descriptors'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['descriptors'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['descriptors'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['missingness'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['missingness'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['missingness'], 'feature_weight': 1.0}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['counts'], 'feature_weight': 0.15}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['counts'], 'feature_weight': 0.5}, {'model': 'logistic', 'C': 0.1, 'scale': 'global', 'features': ['counts'], 'feature_weight': 1.0}]}
for name, content in static_configs.items():
    (replica/"experiments"/name).write_text(json.dumps(content, indent=2))


## Fast exact prediction reproduction

This fits the frozen final model recipe using only training labels and the provided masked training rows. It does not read validation labels. `RUN_FINAL = True` reproduces the prediction by default; set it to False to skip fitting. A fresh output checksum is compared with the observed candidate.

In [16]:
RUN_FINAL = True
if RUN_FINAL:
    for name in ['fullsize-ranking.json','final-candidate.json']:
        shutil.copy2(root/'experiments'/name, replica/'experiments'/name)
    subprocess.run([sys.executable,'make_candidate.py'],cwd=replica,check=True)
    import hashlib
    observed = hashlib.sha256((root/'predictions/part1_fullfit_candidate.csv').read_bytes()).hexdigest()
    reproduced = hashlib.sha256((replica/'predictions/part1_fullfit_candidate.csv').read_bytes()).hexdigest()
    assert observed == reproduced, (observed,reproduced)
    print('Exact prediction checksum matched:', observed)


{
  "created_at": "2026-11-05T04:18:52.504508+00:00",
  "selected": {
    "name": "balanced_ensemble",
    "config": {
      "model": "ensemble",
      "members": [
        {
          "model": "pseudo",
          "teacher": {
            "model": "cluster",
            "cluster": "hdbscan",
            "min_cluster": 5,
            "pca": 25
          },
          "student": "logistic",
          "C": 0.3,
          "gamma": 0.001,
          "scale": "global",
          "pseudo_weight": 0.05,
          "confidence": 0.95,
          "labelwise": true,
          "consensus": true,
          "features": [
            "fingerprint"
          ],
          "feature_weight": 0.5
        },
        {
          "model": "svc",
          "C": 10,
          "gamma": 0.001,
          "scale": "standard",
          "features": [
            "drugs",
            "criteria"
          ],
          "feature_weight": 0.5
        },
        {
          "model": "svc",
          "C": 10,
          "gamma

## Full search replay

The executable command list below covers all logged search stages. Repeated seeds and adaptive stage configuration files are retained in the evidence. For an exact historical replay, copy the recorded JSON input/configuration files; do not re-select model families after seeing external validation. Nested and leave-group-out audits can be run separately. `finalize.py` deliberately refuses a second external validation comparison when its result already exists.

In [17]:
RUN_SEARCH = False
commands = [['experiment.py'], ['prepare_refinement.py'], ['experiment.py', '--stage', 'refine', '--configs', 'experiments/refine-input.json'], ['experiment.py', '--stage', 'features', '--configs', 'experiments/features-input.json'], ['prepare_consensus.py'], ['experiment.py', '--stage', 'consensus', '--configs', 'experiments/consensus-input.json'], ['robustness.py'], ['prepare_metric.py'], ['experiment.py', '--stage', 'metric', '--configs', 'experiments/metric-input.json'], ['experiment.py', '--stage', 'ssl_features', '--configs', 'experiments/ssl-features-input.json', '--seed', '20261104'], ['experiment.py', '--stage', 'ssl_features', '--configs', 'experiments/ssl-features-input.json', '--seed', '20261105'], ['experiment.py', '--stage', 'ssl_features', '--configs', 'experiments/ssl-features-input.json', '--seed', '20261106'], ['experiment.py', '--stage', 'phase2_repeat', '--configs', 'experiments/phase2-repeat-input.json', '--seed', '20261105'], ['experiment.py', '--stage', 'phase2_repeat', '--configs', 'experiments/phase2-repeat-input.json', '--seed', '20261106'], ['experiment.py', '--stage', 'dictionary', '--configs', 'experiments/dictionary-input.json'], ['experiment.py', '--stage', 'phase2', '--configs', 'experiments/phase2-input.json']]
if RUN_SEARCH:
    for command in commands:
        subprocess.run([sys.executable,*command],cwd=replica,check=True)


In [18]:
RUN_AUDITS = False
if RUN_AUDITS:
    shutil.copytree(root/'experiments',replica/'experiments',dirs_exist_ok=True)
    for script in ['verify_engine.py','audit_candidates.py','fullsize_cv.py']:
        subprocess.run([sys.executable,script],cwd=replica,check=True)


## Exact final validation evaluation

At the user’s request, the frozen full-fit candidate was evaluated: sample-F1 0.85043, macro-F1 0.68841. The script below reproduces that measurement without changing the model or threshold and verifies the original test predictions. It is saved here but is not automatically executed.

In [ ]:
evaluation_source = '"""Evaluate the already-selected full-fit candidate, without retuning or validation fitting."""\nfrom pathlib import Path\nimport json, hashlib, datetime\nimport numpy as np\nimport pandas as pd\nfrom threadpoolctl import threadpool_limits\nimport experiment as e\n\nmanifest_path=Path(\'experiments/final-candidate.json\')\nmanifest=json.loads(manifest_path.read_text())\nassert hashlib.sha256(Path(\'experiment.py\').read_bytes()).hexdigest()==manifest[\'engine_sha256\']\nassert hashlib.sha256(Path(manifest[\'submission\']).read_bytes()).hexdigest()==manifest[\'submission_sha256\']\nselected=manifest[\'selected\']\nval=pd.read_csv(\'data/part1_val.csv\',usecols=lambda c:c not in e.LABELS)\ntest=pd.read_csv(\'data/part1_test.csv\')\nunlabeled=e.DATA.iloc[e.UNLAB].drop(columns=e.LABELS)\nquery=pd.concat([val,test,unlabeled],ignore_index=True)\nwith threadpool_limits(limits=1):\n    p=e.predict(np.arange(len(e.Y)),query,selected[\'config\'])\n# Check this is the same candidate that generated the delivered Kaggle predictions.\nsaved=pd.read_csv(\'predictions/fullfit_test_probabilities.csv\')\nnp.testing.assert_allclose(p[len(val):len(val)+len(test)],saved[[\'p_\'+l for l in e.LABELS]].to_numpy(),rtol=1e-10,atol=1e-12)\ntest_predictions=e.decisions(p[len(val):len(val)+len(test)],selected[\'rule\'])\nstrings=[\' \'.join(l for l,v in zip(e.LABELS,row) if v) for row in test_predictions]\nassert strings==pd.read_csv(manifest[\'submission\']).Predicted.tolist()\n# Validation labels are accessed only after fitting and prediction.\ny=pd.read_csv(\'data/part1_val.csv\',usecols=e.LABELS)[e.LABELS].to_numpy(int)\nvp=p[:len(val)];hard=e.decisions(vp,selected[\'rule\'])\nresult={\n    \'evaluated_at\':datetime.datetime.now(datetime.timezone.utc).isoformat(),\n    \'candidate\':selected[\'name\'],\'n_validation\':len(val),\n    \'rule\':selected[\'rule\'],\'config\':selected[\'config\'],\n    **e.metrics(y,vp,selected[\'rule\']),\n    \'training_gold_rows\':len(e.Y),\'validation_used_for_fitting\':False,\n    \'settings_or_threshold_changed\':False,\'same_test_probabilities_verified\':True,\n    \'engine_sha256\':manifest[\'engine_sha256\'],\'submission_sha256\':manifest[\'submission_sha256\'],\n    \'context\':\'User requested direct evaluation of the frozen full-fit candidate. This is a second validation evaluation occasion, with no retuning. Earlier bagged-model validation results remain separate.\'\n}\ne.save_json(\'experiments/final-validation.json\',result)\nout=pd.DataFrame(vp,columns=[\'p_\'+l for l in e.LABELS]);out.insert(0,\'ID\',val.ID.to_numpy())\nfor j,l in enumerate(e.LABELS):out[\'pred_\'+l]=hard[:,j];out[\'true_\'+l]=y[:,j]\nout[\'sample_f1\']=2*(y*hard).sum(1)/(y.sum(1)+hard.sum(1))\nout.to_csv(\'predictions/fullfit_validation_audit.csv\',index=False)\nmanifest[\'validation_status\']=\'Evaluated on all 78 validation rows at user request, with model settings and threshold unchanged.\'\nmanifest[\'validation_metrics\']={k:result[k] for k in [\'sample_f1\',\'macro_f1\']+[\'f1_\'+l for l in e.LABELS]}\nmanifest[\'validation_result\']=\'experiments/final-validation.json\'\ne.save_json(manifest_path,manifest)\nprint(json.dumps(result,indent=2))\n'
(replica / "evaluate_final_validation.py").write_text(evaluation_source)
